In [3]:
import requests
import pandas as pd
import numpy as np
import json
import time

print("Environment ready")

Environment ready


In [4]:
import requests
import time

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": "cyberattack",
    "mode": "artlist",
    "format": "json",
    "maxrecords": 25,
    "timespan": "1week"
}

headers = {
    "User-Agent": "EuropeanSecurityMonitor/1.0"
}

def gdelt_request(url, params, headers, max_retries=3):
    wait_time = 10

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )

        print(
            f"Attempt {attempt + 1} | "
            f"Status: {response.status_code}"
        )

        if response.status_code == 200:
            print("Request successful")
            return response

        elif response.status_code == 429:
            if attempt < max_retries - 1:
                print(
                    f"Rate limit active. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)
                wait_time *= 2

        else:
            print(response.text[:300])
            response.raise_for_status()

    print("Maximum retries reached.")
    return None

In [5]:
print("GDELT API client ready")

GDELT API client ready


## 2. Security domain taxonomy

In [6]:
SECURITY_DOMAINS = {
    "Defence & Military": [
        "military",
        "defence",
        "defense",
        "army",
        "armed forces",
        "missile",
        "weapon",
        "weapons",
        "air defence",
        "military exercise",
        "troops"
    ],

    "Cybersecurity": [
        "cyberattack",
        "cyber attack",
        "ransomware",
        "malware",
        "cybersecurity",
        "data breach",
        "hacking"
    ],

    "Energy Security": [
        "energy security",
        "gas supply",
        "oil supply",
        "pipeline",
        "electricity grid",
        "energy infrastructure"
    ],

    "Sanctions & Economic Security": [
        "sanctions",
        "export controls",
        "economic sanctions",
        "trade restrictions",
        "asset freeze"
    ],

    "Conflict & Geopolitical Tensions": [
        "conflict",
        "war",
        "invasion",
        "border tensions",
        "military escalation",
        "ceasefire",
        "hostilities"
    ]
}

In [7]:
for domain, keywords in SECURITY_DOMAINS.items():
    print(domain)
    print("  ", keywords)

Defence & Military
   ['military', 'defence', 'defense', 'army', 'armed forces', 'missile', 'weapon', 'weapons', 'air defence', 'military exercise', 'troops']
Cybersecurity
   ['cyberattack', 'cyber attack', 'ransomware', 'malware', 'cybersecurity', 'data breach', 'hacking']
Energy Security
   ['energy security', 'gas supply', 'oil supply', 'pipeline', 'electricity grid', 'energy infrastructure']
Sanctions & Economic Security
   ['sanctions', 'export controls', 'economic sanctions', 'trade restrictions', 'asset freeze']
Conflict & Geopolitical Tensions
   ['conflict', 'war', 'invasion', 'border tensions', 'military escalation', 'ceasefire', 'hostilities']


## 3. Rule-Based Security Classification

In [8]:
def classify_security_domain(text, domains):
    """
    Classifies a text into one or more security domains
    based on keyword matches.
    """
    
    if not isinstance(text, str):
        return ["Unclassified"]

    text = text.lower()

    matches = []

    for domain, keywords in domains.items():
        for keyword in keywords:
            if keyword.lower() in text:
                matches.append(domain)
                break

    if not matches:
        return ["Unclassified"]

    return matches

In [9]:
test_text = "Estonia reports major cyberattack against government systems"

classify_security_domain(
    test_text,
    SECURITY_DOMAINS
)

['Cybersecurity']

In [10]:
test_articles = [
    "Estonia reports major cyberattack against government systems",
    "Poland announces new air defence procurement programme",
    "European Union approves new sanctions against Russia",
    "Damage to gas pipeline raises energy security concerns",
    "Military escalation continues near the Ukrainian border",
    "European leaders meet in Brussels to discuss migration"
]

for article in test_articles:
    classification = classify_security_domain(
        article,
        SECURITY_DOMAINS
    )
    
    print(article)
    print("Classification:", classification)
    print("-" * 80)

Estonia reports major cyberattack against government systems
Classification: ['Cybersecurity']
--------------------------------------------------------------------------------
Poland announces new air defence procurement programme
Classification: ['Defence & Military']
--------------------------------------------------------------------------------
European Union approves new sanctions against Russia
Classification: ['Sanctions & Economic Security']
--------------------------------------------------------------------------------
Damage to gas pipeline raises energy security concerns
Classification: ['Energy Security']
--------------------------------------------------------------------------------
Military escalation continues near the Ukrainian border
Classification: ['Defence & Military', 'Conflict & Geopolitical Tensions']
--------------------------------------------------------------------------------
European leaders meet in Brussels to discuss migration
Classification: ['Unclassi

In [11]:
df_test = pd.DataFrame({
    "title": test_articles
})

df_test

,title
0,Estonia reports major cyberattack against gove...
1,Poland announces new air defence procurement p...
2,European Union approves new sanctions against ...
3,Damage to gas pipeline raises energy security ...
4,Military escalation continues near the Ukraini...
5,European leaders meet in Brussels to discuss m...


In [12]:
df_test["security_domain"] = df_test["title"].apply(
    lambda x: classify_security_domain(
        x,
        SECURITY_DOMAINS
    )
)

df_test

,title,security_domain
0,Estonia reports major cyberattack against gove...,[Cybersecurity]
1,Poland announces new air defence procurement p...,[Defence & Military]
2,European Union approves new sanctions against ...,[Sanctions & Economic Security]
3,Damage to gas pipeline raises energy security ...,[Energy Security]
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T..."
5,European leaders meet in Brussels to discuss m...,[Unclassified]


In [13]:
def find_security_keywords(text, domains):
    """
    Returns the security keywords detected in a text.
    """
    
    if not isinstance(text, str):
        return []

    text = text.lower()

    detected_keywords = []

    for domain, keywords in domains.items():
        for keyword in keywords:
            if keyword.lower() in text:
                detected_keywords.append(keyword)

    return list(set(detected_keywords))

In [14]:
df_test["matched_keywords"] = df_test["title"].apply(
    lambda x: find_security_keywords(
        x,
        SECURITY_DOMAINS
    )
)

df_test

,title,security_domain,matched_keywords
0,Estonia reports major cyberattack against gove...,[Cybersecurity],[cyberattack]
1,Poland announces new air defence procurement p...,[Defence & Military],"[air defence, defence]"
2,European Union approves new sanctions against ...,[Sanctions & Economic Security],[sanctions]
3,Damage to gas pipeline raises energy security ...,[Energy Security],"[pipeline, energy security]"
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T...","[military escalation, military]"
5,European leaders meet in Brussels to discuss m...,[Unclassified],[]


## 4. Country Entity Extraction with spaCy

In [15]:
import sys

print(sys.executable)

c:\Users\cl_am\AppData\Local\Programs\Python\Python314\python.exe


In [16]:
EUROPEAN_COUNTRIES = [
    "Albania",
    "Austria",
    "Andorra",
    "Belarus",
    "Belgium",
    "Bosnia and Herzegovina",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czechia",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Iceland",
    "Ireland",
    "Italy",
    "Kosovo",
    "Latvia",
    "Liechtenstein",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Moldova",
    "Monaco",
    "Montenegro",
    "Netherlands",
    "North Macedonia",
    "Norway",
    "Poland",
    "Portugal",
    "Romania",
    "San Marino",
    "Serbia",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "Switzerland",
    "Türkiye",
    "Ukraine",
    "United Kingdom"
    "Vatican City",
]

STRATEGIC_NEIGHBOURS = [
    "Russia",
    "Georgia",
    "Armenia",
    "Azerbaijan"
]

MONITORED_COUNTRIES = EUROPEAN_COUNTRIES + STRATEGIC_NEIGHBOURS

print(len(MONITORED_COUNTRIES))

49


In [17]:

COUNTRY_ALIASES = {
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Britain": "United Kingdom",
    "British": "United Kingdom",

    "Turkey": "Türkiye",
    "Turkish": "Türkiye",

    "Czech Republic": "Czechia",

    "Macedonia": "North Macedonia",

    "Russian": "Russia",
    "Russians": "Russia",

    "Ukrainian": "Ukraine",
    "Ukrainians": "Ukraine",

    "Belarusian": "Belarus",

    "Polish": "Poland",
    "German": "Germany",
    "French": "France",
    "Spanish": "Spain",
    "Italian": "Italy",
    "Estonian": "Estonia",
    "Latvian": "Latvia",
    "Lithuanian": "Lithuania",
    "Finnish": "Finland",
    "Swedish": "Sweden",
    "Norwegian": "Norway",
    "Danish": "Denmark"
}

In [18]:
import re

def extract_countries(text):
    """
    Detects monitored countries mentioned in text
    using country names and aliases.
    """

    if not isinstance(text, str):
        return []

    detected = []

    # Direct country names
    for country in MONITORED_COUNTRIES:

        pattern = rf"\b{re.escape(country)}\b"

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            detected.append(country)

    # Country aliases
    for alias, country in COUNTRY_ALIASES.items():

        pattern = rf"\b{re.escape(alias)}\b"

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            detected.append(country)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(detected))

In [19]:
country_tests = [
    "Poland announces new air defence procurement programme",
    "Estonia reports major cyberattack against government systems",
    "Germany and France discuss new defence cooperation",
    "British forces deploy to Poland",
    "Russia launches new attacks against Ukraine",
    "NATO officials meet in Brussels",
    "European leaders discuss defence spending"
]

for article in country_tests:

    countries = extract_countries(article)

    print(article)
    print("Countries:", countries)
    print("-" * 80)

Poland announces new air defence procurement programme
Countries: ['Poland']
--------------------------------------------------------------------------------
Estonia reports major cyberattack against government systems
Countries: ['Estonia']
--------------------------------------------------------------------------------
Germany and France discuss new defence cooperation
Countries: ['France', 'Germany']
--------------------------------------------------------------------------------
British forces deploy to Poland
Countries: ['Poland', 'United Kingdom']
--------------------------------------------------------------------------------
Russia launches new attacks against Ukraine
Countries: ['Ukraine', 'Russia']
--------------------------------------------------------------------------------
NATO officials meet in Brussels
Countries: []
--------------------------------------------------------------------------------
European leaders discuss defence spending
Countries: []
------------------

In [20]:
df_test["countries"] = df_test["title"].apply(
    extract_countries
)

df_test

,title,security_domain,matched_keywords,countries
0,Estonia reports major cyberattack against gove...,[Cybersecurity],[cyberattack],[Estonia]
1,Poland announces new air defence procurement p...,[Defence & Military],"[air defence, defence]",[Poland]
2,European Union approves new sanctions against ...,[Sanctions & Economic Security],[sanctions],[Russia]
3,Damage to gas pipeline raises energy security ...,[Energy Security],"[pipeline, energy security]",[]
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T...","[military escalation, military]",[Ukraine]
5,European leaders meet in Brussels to discuss m...,[Unclassified],[],[]


## 5. GDELT API Data Acquisition

In [21]:
import requests

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": "cyberattack",
    "mode": "artlist",
    "format": "json",
    "maxrecords": 25,
    "timespan": "1week"
}

headers = {
    "User-Agent": "EuropeanSecurityMonitor/1.0"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [22]:
import requests

last_update_url = "https://data.gdeltproject.org/gdeltv2/lastupdate.txt"

response = requests.get(
    last_update_url,
    timeout=30
)

print("Status code:", response.status_code)
print(response.text)

Status code: 200
84575 a7b9003328b2ead847de097a3bfe0d59 http://data.gdeltproject.org/gdeltv2/20260825114500.export.CSV.zip
108141 821f4434a2228681a70fb3a51aca07d9 http://data.gdeltproject.org/gdeltv2/20260825114500.mentions.CSV.zip
5402174 147077ece11e5372e6b003931831381e http://data.gdeltproject.org/gdeltv2/20260825114500.gkg.csv.zip



In [23]:
lines = response.text.strip().splitlines()

export_line = lines[0]

export_url = export_line.split()[-1]

print("Latest GDELT Events file:")
print(export_url)

Latest GDELT Events file:
http://data.gdeltproject.org/gdeltv2/20260825114500.export.CSV.zip


In [24]:
from pathlib import Path

# Detect project root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data folder:", raw_dir)

Project root: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor
Raw data folder: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw


In [25]:
# Use HTTPS
export_url = export_url.replace("http://", "https://")

file_name = export_url.split("/")[-1]
zip_path = raw_dir / file_name

gdelt_response = requests.get(
    export_url,
    timeout=60
)

gdelt_response.raise_for_status()

with open(zip_path, "wb") as file:
    file.write(gdelt_response.content)

print("Downloaded:")
print(zip_path)
print("File size:", round(zip_path.stat().st_size / 1024, 2), "KB")

Downloaded:
c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw\20260825114500.export.CSV.zip
File size: 82.59 KB


## 6. Load GDELT Events

In [26]:
GDELT_EVENT_COLUMNS = [
    "GLOBALEVENTID",
    "SQLDATE",
    "MonthYear",
    "Year",
    "FractionDate",
    "Actor1Code",
    "Actor1Name",
    "Actor1CountryCode",
    "Actor1KnownGroupCode",
    "Actor1EthnicCode",
    "Actor1Religion1Code",
    "Actor1Religion2Code",
    "Actor1Type1Code",
    "Actor1Type2Code",
    "Actor1Type3Code",
    "Actor2Code",
    "Actor2Name",
    "Actor2CountryCode",
    "Actor2KnownGroupCode",
    "Actor2EthnicCode",
    "Actor2Religion1Code",
    "Actor2Religion2Code",
    "Actor2Type1Code",
    "Actor2Type2Code",
    "Actor2Type3Code",
    "IsRootEvent",
    "EventCode",
    "EventBaseCode",
    "EventRootCode",
    "QuadClass",
    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",
    "Actor1Geo_Type",
    "Actor1Geo_FullName",
    "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code",
    "Actor1Geo_ADM2Code",
    "Actor1Geo_Lat",
    "Actor1Geo_Long",
    "Actor1Geo_FeatureID",
    "Actor2Geo_Type",
    "Actor2Geo_FullName",
    "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code",
    "Actor2Geo_ADM2Code",
    "Actor2Geo_Lat",
    "Actor2Geo_Long",
    "Actor2Geo_FeatureID",
    "ActionGeo_Type",
    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code",
    "ActionGeo_ADM2Code",
    "ActionGeo_Lat",
    "ActionGeo_Long",
    "ActionGeo_FeatureID",
    "DATEADDED",
    "SOURCEURL"
]

In [27]:
df_gdelt = pd.read_csv(
    zip_path,
    sep="\t",
    header=None,
    names=GDELT_EVENT_COLUMNS,
    compression="zip",
    low_memory=False
)

print("Rows:", len(df_gdelt))
print("Columns:", len(df_gdelt.columns))

df_gdelt.head()

Rows: 1284
Columns: 61


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1319838769,20160827,201608,2016,2016.6493,LKA,SRI LANKA,LKA,NaN,NaN,...,1,Russia,RS,RS,NaN,60.0000,100.00000,RS,20260825114500,https://www.freemalaysiatoday.com/category/bus...
1,1319838770,20250825,202508,2025,2025.6438,NaN,NaN,NaN,NaN,NaN,...,4,"Zeeland, Noord-Brabant, Netherlands",NL,NL06,22372,51.6969,5.67462,-2156284,20260825114500,https://927thevan.com/2026/08/25/more-schools-...
2,1319838771,20250825,202508,2025,2025.6438,NaN,NaN,NaN,NaN,NaN,...,4,"Black River, Manitoba, Canada",CA,CA03,154724,53.4500,-97.66670,-560954,20260825114500,https://927thevan.com/2026/08/25/more-schools-...
3,1319838772,20250825,202508,2025,2025.6438,EDU,UNIVERSITY,NaN,NaN,NaN,...,4,"Zeeland, Noord-Brabant, Netherlands",NL,NL06,22372,51.6969,5.67462,-2156284,20260825114500,https://927thevan.com/2026/08/25/more-schools-...
4,1319838773,20250825,202508,2025,2025.6438,EDU,UNIVERSITY,NaN,NaN,NaN,...,4,"Black River, Manitoba, Canada",CA,CA03,154724,53.4500,-97.66670,-560954,20260825114500,https://927thevan.com/2026/08/25/more-schools-...


## 7. Initial Event Dataset

In [28]:
selected_columns = [
    "GLOBALEVENTID",
    "SQLDATE",
    "Actor1Name",
    "Actor1CountryCode",
    "Actor2Name",
    "Actor2CountryCode",
    "EventCode",
    "EventRootCode",
    "QuadClass",
    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",
    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "ActionGeo_Lat",
    "ActionGeo_Long",
    "SOURCEURL"
]

df_events = df_gdelt[selected_columns].copy()

In [29]:
df_events = df_events.rename(columns={
    "GLOBALEVENTID": "event_id",
    "SQLDATE": "event_date",
    "Actor1Name": "actor1",
    "Actor1CountryCode": "actor1_country",
    "Actor2Name": "actor2",
    "Actor2CountryCode": "actor2_country",
    "EventCode": "event_code",
    "EventRootCode": "event_root_code",
    "QuadClass": "quad_class",
    "GoldsteinScale": "goldstein_scale",
    "NumMentions": "num_mentions",
    "NumSources": "num_sources",
    "NumArticles": "num_articles",
    "AvgTone": "avg_tone",
    "ActionGeo_FullName": "location",
    "ActionGeo_CountryCode": "location_country",
    "ActionGeo_Lat": "latitude",
    "ActionGeo_Long": "longitude",
    "SOURCEURL": "source_url"
})

In [30]:
df_events["event_date"] = pd.to_datetime(
    df_events["event_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

In [31]:
print("Rows:", len(df_events))
print("Columns:", len(df_events.columns))
print("Date range:")
print(df_events["event_date"].min(), "→", df_events["event_date"].max())

df_events.head(10)

Rows: 1284
Columns: 19
Date range:
2016-08-27 00:00:00 → 2026-08-25 00:00:00


,event_id,event_date,actor1,actor1_country,actor2,actor2_country,event_code,event_root_code,quad_class,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,location,location_country,latitude,longitude,source_url
0,1319838769,2016-08-27,SRI LANKA,LKA,INVESTOR,NaN,13,1,1,0.4,4,1,4,-0.531915,Russia,RS,60.0000,100.00000,https://www.freemalaysiatoday.com/category/bus...
1,1319838770,2025-08-25,NaN,NaN,UNIVERSITY,NaN,42,4,1,1.9,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
2,1319838771,2025-08-25,NaN,NaN,UNIVERSITY,NaN,42,4,1,1.9,8,2,8,-0.854701,"Black River, Manitoba, Canada",CA,53.4500,-97.66670,https://927thevan.com/2026/08/25/more-schools-...
3,1319838772,2025-08-25,UNIVERSITY,NaN,NaN,NaN,43,4,1,2.8,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
4,1319838773,2025-08-25,UNIVERSITY,NaN,NaN,NaN,43,4,1,2.8,8,2,8,-0.854701,"Black River, Manitoba, Canada",CA,53.4500,-97.66670,https://927thevan.com/2026/08/25/more-schools-...
5,1319838774,2025-08-25,UNIVERSITY,NaN,NETHERLANDS,NLD,43,4,1,2.8,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
6,1319838775,2025-08-25,UNIVERSITY,NaN,NETHERLANDS,NLD,43,4,1,2.8,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
7,1319838776,2025-08-25,NAVY,NaN,RUSSIAN,RUS,90,9,2,-2.0,7,1,7,-0.511945,"Kremlin, Moskva, Russia",RS,55.7522,37.61560,https://www.express.co.uk/news/world/2242228/z...
8,1319838777,2025-08-25,NAVY,NaN,RUSSIAN,RUS,90,9,2,-2.0,1,1,1,-0.511945,United Kingdom,UK,54.0000,-4.00000,https://www.express.co.uk/news/world/2242228/z...
9,1319838778,2025-08-25,NETHERLANDS,NLD,UNIVERSITY,NaN,42,4,1,1.9,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...


In [32]:
print("df_gdelt rows:", len(df_gdelt))
print("df_events rows:", len(df_events))

print("\ndf_gdelt shape:")
print(df_gdelt.shape)

print("\ndf_events shape:")
print(df_events.shape)

df_gdelt rows: 1284
df_events rows: 1284

df_gdelt shape:
(1284, 61)

df_events shape:
(1284, 19)


In [33]:
df_events.head()

,event_id,event_date,actor1,actor1_country,actor2,actor2_country,event_code,event_root_code,quad_class,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,location,location_country,latitude,longitude,source_url
0,1319838769,2016-08-27,SRI LANKA,LKA,INVESTOR,NaN,13,1,1,0.4,4,1,4,-0.531915,Russia,RS,60.0000,100.00000,https://www.freemalaysiatoday.com/category/bus...
1,1319838770,2025-08-25,NaN,NaN,UNIVERSITY,NaN,42,4,1,1.9,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
2,1319838771,2025-08-25,NaN,NaN,UNIVERSITY,NaN,42,4,1,1.9,8,2,8,-0.854701,"Black River, Manitoba, Canada",CA,53.4500,-97.66670,https://927thevan.com/2026/08/25/more-schools-...
3,1319838772,2025-08-25,UNIVERSITY,NaN,NaN,NaN,43,4,1,2.8,4,2,4,-0.854701,"Zeeland, Noord-Brabant, Netherlands",NL,51.6969,5.67462,https://927thevan.com/2026/08/25/more-schools-...
4,1319838773,2025-08-25,UNIVERSITY,NaN,NaN,NaN,43,4,1,2.8,8,2,8,-0.854701,"Black River, Manitoba, Canada",CA,53.4500,-97.66670,https://927thevan.com/2026/08/25/more-schools-...


## 8. Geographic Coverage Inspection

In [34]:
print("Actor 1 country codes:")
print(
    sorted(
        df_events["actor1_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

print("\nActor 2 country codes:")
print(
    sorted(
        df_events["actor2_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

print("\nLocation country codes:")
print(
    sorted(
        df_events["location_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

Actor 1 country codes:
['AFG', 'AFR', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'BEL', 'BGD', 'BOL', 'BRA', 'BRN', 'CAN', 'CHE', 'CHN', 'COD', 'COL', 'CUB', 'CZE', 'DEU', 'DNK', 'EGY', 'ESP', 'EUR', 'FIN', 'FRA', 'GBR', 'GMB', 'GRC', 'GTM', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'ISR', 'ITA', 'JOR', 'JPN', 'KOR', 'KWT', 'LBN', 'LBY', 'LKA', 'LUX', 'MCO', 'MDA', 'MKD', 'MMR', 'MYS', 'NGA', 'NLD', 'NZL', 'PAK', 'PHL', 'PNG', 'PRK', 'PSE', 'QAT', 'RUS', 'RWA', 'SAU', 'SWE', 'SYR', 'THA', 'TON', 'TUR', 'TUV', 'UGA', 'UKR', 'USA', 'VEN', 'VNM', 'VUT', 'WST', 'ZAF', 'ZMB']

Actor 2 country codes:
['AFG', 'AFR', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'BEL', 'BGD', 'BHR', 'BOL', 'BRA', 'CAN', 'CHL', 'CHN', 'CMR', 'COD', 'CUB', 'DEU', 'DNK', 'EGY', 'EUR', 'FIN', 'FRA', 'GBR', 'GMB', 'GRC', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISR', 'ITA', 'JOR', 'JPN', 'KOR', 'KWT', 'LBN', 'MDA', 'MMR', 'MWI', 'MYS', 'NLD', 'NOR', 'PAK', 'PAN', 'PHL', 'PRK', 'PRT', 'PSE', 'QAT', 'RUS', 'RWA', 'SAU', 'SEA', 'SGP', 'SWE', 'SYR',

In [35]:
df_events[
    [
        "actor1",
        "actor1_country",
        "actor2",
        "actor2_country",
        "location",
        "location_country"
    ]
].head(20)

,actor1,actor1_country,actor2,actor2_country,location,location_country
0,SRI LANKA,LKA,INVESTOR,NaN,Russia,RS
1,NaN,NaN,UNIVERSITY,NaN,"Zeeland, Noord-Brabant, Netherlands",NL
2,NaN,NaN,UNIVERSITY,NaN,"Black River, Manitoba, Canada",CA
3,UNIVERSITY,NaN,NaN,NaN,"Zeeland, Noord-Brabant, Netherlands",NL
4,UNIVERSITY,NaN,NaN,NaN,"Black River, Manitoba, Canada",CA
5,UNIVERSITY,NaN,NETHERLANDS,NLD,"Zeeland, Noord-Brabant, Netherlands",NL
6,UNIVERSITY,NaN,NETHERLANDS,NLD,"Zeeland, Noord-Brabant, Netherlands",NL
7,NAVY,NaN,RUSSIAN,RUS,"Kremlin, Moskva, Russia",RS
8,NAVY,NaN,RUSSIAN,RUS,United Kingdom,UK
9,NETHERLANDS,NLD,UNIVERSITY,NaN,"Zeeland, Noord-Brabant, Netherlands",NL


## 9. European & Strategic Event Filtering

In [36]:
MONITORED_ISO3 = {
    "ALB",  # Albania
    "AUT",  # Austria
    "BLR",  # Belarus
    "BEL",  # Belgium
    "BIH",  # Bosnia and Herzegovina
    "BGR",  # Bulgaria
    "HRV",  # Croatia
    "CYP",  # Cyprus
    "CZE",  # Czechia
    "DNK",  # Denmark
    "EST",  # Estonia
    "FIN",  # Finland
    "FRA",  # France
    "DEU",  # Germany
    "GRC",  # Greece
    "HUN",  # Hungary
    "ISL",  # Iceland
    "IRL",  # Ireland
    "ITA",  # Italy
    "LVA",  # Latvia
    "LTU",  # Lithuania
    "LUX",  # Luxembourg
    "MLT",  # Malta
    "MDA",  # Moldova
    "MNE",  # Montenegro
    "NLD",  # Netherlands
    "MKD",  # North Macedonia
    "NOR",  # Norway
    "POL",  # Poland
    "PRT",  # Portugal
    "ROU",  # Romania
    "SRB",  # Serbia
    "SVK",  # Slovakia
    "SVN",  # Slovenia
    "ESP",  # Spain
    "SWE",  # Sweden
    "CHE",  # Switzerland
    "TUR",  # Türkiye
    "UKR",  # Ukraine
    "GBR",  # United Kingdom

    # Strategic neighbourhood
    "RUS",  # Russia
    "GEO",  # Georgia
    "ARM",  # Armenia
    "AZE"   # Azerbaijan
}

In [37]:
df_events["location_countries"] = (
    df_events["location"]
    .apply(extract_countries)
)

In [38]:
df_events[
    [
        "location",
        "location_country",
        "location_countries"
    ]
].head(20)

,location,location_country,location_countries
0,Russia,RS,[Russia]
1,"Zeeland, Noord-Brabant, Netherlands",NL,[Netherlands]
2,"Black River, Manitoba, Canada",CA,[]
3,"Zeeland, Noord-Brabant, Netherlands",NL,[Netherlands]
4,"Black River, Manitoba, Canada",CA,[]
5,"Zeeland, Noord-Brabant, Netherlands",NL,[Netherlands]
6,"Zeeland, Noord-Brabant, Netherlands",NL,[Netherlands]
7,"Kremlin, Moskva, Russia",RS,[Russia]
8,United Kingdom,UK,[]
9,"Zeeland, Noord-Brabant, Netherlands",NL,[Netherlands]


In [39]:
actor1_match = (
    df_events["actor1_country"]
    .isin(MONITORED_ISO3)
)

actor2_match = (
    df_events["actor2_country"]
    .isin(MONITORED_ISO3)
)

location_match = (
    df_events["location_countries"]
    .apply(lambda x: len(x) > 0)
)

df_europe = df_events[
    actor1_match |
    actor2_match |
    location_match
].copy()

print("Global events:", len(df_events))
print("European / strategic events:", len(df_europe))

print(
    "Share retained:",
    round(
        len(df_europe) / len(df_events) * 100,
        2
    ),
    "%"
)

Global events: 1284
European / strategic events: 467
Share retained: 36.37 %


In [40]:
df_europe[
    [
        "event_date",
        "actor1",
        "actor1_country",
        "actor2",
        "actor2_country",
        "location",
        "location_countries"
    ]
].head(20)

,event_date,actor1,actor1_country,actor2,actor2_country,location,location_countries
0,2016-08-27,SRI LANKA,LKA,INVESTOR,NaN,Russia,[Russia]
1,2025-08-25,NaN,NaN,UNIVERSITY,NaN,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
3,2025-08-25,UNIVERSITY,NaN,NaN,NaN,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
5,2025-08-25,UNIVERSITY,NaN,NETHERLANDS,NLD,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
6,2025-08-25,UNIVERSITY,NaN,NETHERLANDS,NLD,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
7,2025-08-25,NAVY,NaN,RUSSIAN,RUS,"Kremlin, Moskva, Russia",[Russia]
8,2025-08-25,NAVY,NaN,RUSSIAN,RUS,United Kingdom,[]
9,2025-08-25,NETHERLANDS,NLD,UNIVERSITY,NaN,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
10,2025-08-25,NETHERLANDS,NLD,UNIVERSITY,NaN,"Zeeland, Noord-Brabant, Netherlands",[Netherlands]
12,2026-07-26,UNITED KINGDOM,GBR,KING,NaN,"Balmoral Castle, Aberdeenshire, United Kingdom",[]


In [41]:
LOCATION_COUNTRY_ALIASES = {
    "Turkey": "Türkiye",
    "Czech Republic": "Czechia",
    "Russian Federation": "Russia",
    "Vatican": "Vatican City"
}


def extract_gdelt_location_country(location):
    """
    Extracts the country from a GDELT geographic location.
    GDELT locations usually end with the country name.
    """

    if not isinstance(location, str):
        return []

    # Last component of the GDELT location
    country_candidate = location.split(",")[-1].strip()

    # Normalise aliases
    country_candidate = LOCATION_COUNTRY_ALIASES.get(
        country_candidate,
        country_candidate
    )

    if country_candidate in MONITORED_COUNTRIES:
        return [country_candidate]

    return []

In [42]:
df_events["location_countries"] = (
    df_events["location"]
    .apply(extract_gdelt_location_country)
)

In [43]:
test_locations = [
    "Paris, France (general), France",
    "Port-Of-Spain, Port-of-Spain, Trinidad And Tobago",
    "Scottish Highlands, Highland, United Kingdom",
    "Monaco",
    "Nagoya, Aichi, Japan",
    "Kyiv, Ukraine"
]

for location in test_locations:
    print(
        location,
        "→",
        extract_gdelt_location_country(location)
    )

Paris, France (general), France → ['France']
Port-Of-Spain, Port-of-Spain, Trinidad And Tobago → []
Scottish Highlands, Highland, United Kingdom → []
Monaco → ['Monaco']
Nagoya, Aichi, Japan → []
Kyiv, Ukraine → ['Ukraine']


In [44]:
actor1_match = (
    df_events["actor1_country"]
    .isin(MONITORED_ISO3)
)

actor2_match = (
    df_events["actor2_country"]
    .isin(MONITORED_ISO3)
)

location_match = (
    df_events["location_countries"]
    .apply(lambda x: len(x) > 0)
)

df_europe = df_events[
    actor1_match |
    actor2_match |
    location_match
].copy()

print("Global events:", len(df_events))
print("European / strategic events:", len(df_europe))

print(
    "Share retained:",
    round(
        len(df_europe) / len(df_events) * 100,
        2
    ),
    "%"
)

Global events: 1284
European / strategic events: 462
Share retained: 35.98 %


## 10. CAMEO Event Classification

In [45]:
QUAD_CLASS_LABELS = {
    1: "Verbal Cooperation",
    2: "Material Cooperation",
    3: "Verbal Conflict",
    4: "Material Conflict"
}

df_europe["quad_class_label"] = (
    df_europe["quad_class"]
    .map(QUAD_CLASS_LABELS)
)

quad_distribution = (
    df_europe["quad_class_label"]
    .value_counts()
    .reset_index()
)

quad_distribution.columns = [
    "quad_class",
    "events"
]

quad_distribution

quad_distribution["share_pct"] = (
    quad_distribution["events"]
    / len(df_europe)
    * 100
).round(2)

quad_distribution

,quad_class,events,share_pct
0,Verbal Cooperation,293,63.42
1,Material Cooperation,72,15.58
2,Verbal Conflict,56,12.12
3,Material Conflict,41,8.87


In [46]:
df_europe["goldstein_scale"].describe()

count    462.000000
mean       1.557576
std        4.494515
min      -10.000000
25%       -0.400000
50%        2.800000
75%        4.000000
max        9.000000
Name: goldstein_scale, dtype: float64

In [47]:
print(
    "Average Goldstein:",
    round(
        df_europe["goldstein_scale"].mean(),
        2
    )
)

print(
    "Minimum:",
    df_europe["goldstein_scale"].min()
)

print(
    "Maximum:",
    df_europe["goldstein_scale"].max()
)

Average Goldstein: 1.56
Minimum: -10.0
Maximum: 9.0


In [48]:
CAMEO_ROOT_CODES = {
    1: "Make Public Statement",
    2: "Appeal",
    3: "Express Intent to Cooperate",
    4: "Consult",
    5: "Engage in Diplomatic Cooperation",
    6: "Engage in Material Cooperation",
    7: "Provide Aid",
    8: "Yield",
    9: "Investigate",
    10: "Demand",
    11: "Disapprove",
    12: "Reject",
    13: "Threaten",
    14: "Protest",
    15: "Exhibit Force Posture",
    16: "Reduce Relations",
    17: "Coerce",
    18: "Assault",
    19: "Fight",
    20: "Use Unconventional Mass Violence"
}

In [49]:
df_europe["event_root_label"] = (
    df_europe["event_root_code"]
    .map(CAMEO_ROOT_CODES)
)

In [50]:
root_distribution = (
    df_europe["event_root_label"]
    .value_counts()
    .reset_index()
)

root_distribution.columns = [
    "event_type",
    "events"
]

root_distribution.head(15)

,event_type,events
0,Consult,89
1,Make Public Statement,73
2,Engage in Diplomatic Cooperation,62
3,Disapprove,39
4,Express Intent to Cooperate,36
5,Appeal,33
6,Yield,25
7,Fight,23
8,Provide Aid,21
9,Investigate,14


## 11. Security Attention Classification

In [51]:
HIGH_ATTENTION_ROOTS = {
    13,  # Threaten
    15,  # Exhibit Force Posture
    16,  # Reduce Relations
    17,  # Coerce
}

CRITICAL_ATTENTION_ROOTS = {
    18,  # Assault
    19,  # Fight
    20   # Use Unconventional Mass Violence
}

MEDIUM_ATTENTION_ROOTS = {
    10,  # Demand
    11,  # Disapprove
    12,  # Reject
    14   # Protest
}



In [52]:
def classify_attention_level(row):
    
    root_code = row["event_root_code"]
    goldstein = row["goldstein_scale"]

    # Critical conflict events
    if root_code in CRITICAL_ATTENTION_ROOTS:
        return "Critical"

    # Strong conflict / coercive behaviour
    if root_code in HIGH_ATTENTION_ROOTS:
        return "High"

    # Moderate political tension
    if root_code in MEDIUM_ATTENTION_ROOTS:
        return "Medium"

    # Additional safeguard:
    # very negative Goldstein events deserve attention
    if pd.notna(goldstein) and goldstein <= -5:
        return "High"

    return "Low"

In [53]:
df_europe["attention_level"] = df_europe.apply(
    classify_attention_level,
    axis=1
)

In [54]:
attention_distribution = (
    df_europe["attention_level"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"],
        fill_value=0
    )
    .reset_index()
)

attention_distribution.columns = [
    "attention_level",
    "events"
]

attention_distribution["share_pct"] = (
    attention_distribution["events"]
    / len(df_europe)
    * 100
).round(2)

attention_distribution

,attention_level,events,share_pct
0,Critical,27,5.84
1,High,20,4.33
2,Medium,50,10.82
3,Low,365,79.00


In [55]:
df_europe[
    df_europe["attention_level"].isin(
        ["Critical", "High"]
    )
][
    [
        "event_date",
        "actor1",
        "actor2",
        "event_root_label",
        "quad_class_label",
        "goldstein_scale",
        "avg_tone",
        "location",
        "source_url"
    ]
].sort_values(
    "goldstein_scale"
).head(20)

,event_date,actor1,actor2,event_root_label,quad_class_label,goldstein_scale,avg_tone,location,source_url
117,2026-08-25,NaN,GOOGLE,Fight,Material Conflict,-10.0,0.000000,Montenegro,https://www.sunstar.com.ph/davao/minda-rues-tr...
396,2026-08-25,GERMANY,NaN,Fight,Material Conflict,-10.0,-6.637168,"Berlin, Berlin, Germany",https://www.freemalaysiatoday.com/category/wor...
500,2026-08-25,SCOTLAND,NaN,Fight,Material Conflict,-10.0,-1.418440,"West Yorkshire, United Kingdom (general), Unit...",https://www.mirror.co.uk/news/uk-news/minnie-m...
544,2026-08-25,BRITAIN,UKRAINE,Fight,Material Conflict,-10.0,-5.269608,United Kingdom,https://www.lbc.co.uk/article/kremlin-adviser-...
555,2026-08-25,SCOTLAND,GLASGOW,Fight,Material Conflict,-10.0,-2.512563,"Glasgow, Glasgow City, United Kingdom",https://www.glasgowtimes.co.uk/news/scottish-n...
552,2026-08-25,UNITED KINGDOM,NaN,Fight,Material Conflict,-10.0,-8.695652,"Hampshire, Hampshire, United Kingdom",https://wcbm.com/national-headline/did-the-uk-...
548,2026-08-25,UNITED KINGDOM,DETECTIVE,Fight,Material Conflict,-10.0,5.188679,"London, London, City of, United Kingdom",https://www.express.co.uk/showbiz/tv-radio/224...
1030,2026-08-25,RUSSIAN,MOLDOVA,Fight,Material Conflict,-10.0,-7.667732,"Novoshakhtinsk, Rostovskaya Oblast', Russia",https://aa.com.tr/en/russia-ukraine-war/at-lea...
931,2026-08-25,GOOGLE,NaN,Fight,Material Conflict,-10.0,0.000000,Montenegro,https://www.sunstar.com.ph/davao/minda-rues-tr...
1010,2026-08-25,RUSSIAN,NaN,Assault,Material Conflict,-10.0,-6.532663,"Moscow, Moskva, Russia",https://www.lbc.co.uk/article/uk-faces-growing...


In [56]:
attention_metrics = df_europe[
    [
        "num_mentions",
        "num_sources",
        "num_articles",
        "avg_tone"
    ]
].describe()

attention_metrics

,num_mentions,num_sources,num_articles,avg_tone
count,462.000000,462.000000,462.000000,462.000000
mean,3.991342,1.012987,3.800866,-0.680171
std,3.411721,0.113341,2.828304,3.888969
min,1.000000,1.000000,1.000000,-10.073710
25%,2.000000,1.000000,2.000000,-3.365900
50%,2.000000,1.000000,2.000000,-0.511945
75%,6.000000,1.000000,6.000000,2.689076
max,20.000000,2.000000,10.000000,8.333333


In [57]:
print("Correlation matrix:")

df_europe[
    [
        "num_mentions",
        "num_sources",
        "num_articles",
        "avg_tone",
        "goldstein_scale"
    ]
].corr().round(2)

Correlation matrix:


,num_mentions,num_sources,num_articles,avg_tone,goldstein_scale
num_mentions,1.00,0.00,0.94,-0.07,-0.09
num_sources,0.00,1.00,0.01,-0.01,0.02
num_articles,0.94,0.01,1.00,-0.10,-0.10
avg_tone,-0.07,-0.01,-0.10,1.00,0.51
goldstein_scale,-0.09,0.02,-0.10,0.51,1.00


## 12. Security Attention Score

In [58]:
CAMEO_SEVERITY = {
    1: 5,     # Make Public Statement
    2: 5,     # Appeal
    3: 5,     # Express Intent to Cooperate
    4: 5,     # Consult
    5: 5,     # Diplomatic Cooperation

    6: 10,    # Material Cooperation
    7: 10,    # Provide Aid
    8: 15,    # Yield

    9: 20,    # Investigate

    10: 35,   # Demand
    11: 35,   # Disapprove
    12: 40,   # Reject
    13: 60,   # Threaten
    14: 45,   # Protest
    15: 70,   # Exhibit Force Posture
    16: 65,   # Reduce Relations
    17: 75,   # Coerce

    18: 90,   # Assault
    19: 95,   # Fight
    20: 100   # Unconventional Mass Violence
}

df_europe["cameo_severity"] = (
    df_europe["event_root_code"]
    .map(CAMEO_SEVERITY)
    .fillna(0)
)

In [59]:
df_europe["goldstein_conflict"] = (
    (10 - df_europe["goldstein_scale"]) / 20 * 100
).clip(0, 100)

In [60]:
df_europe[
    [
        "goldstein_scale",
        "goldstein_conflict"
    ]
].head(10)

,goldstein_scale,goldstein_conflict
0,0.4,48.0
1,1.9,40.5
3,2.8,36.0
5,2.8,36.0
6,2.8,36.0
7,-2.0,60.0
8,-2.0,60.0
9,1.9,40.5
10,1.9,40.5
12,1.0,45.0


In [61]:
df_europe["event_severity_score"] = (
    0.60 * df_europe["cameo_severity"] +
    0.40 * df_europe["goldstein_conflict"]
)

In [62]:
import numpy as np

mentions_reference = df_europe["num_mentions"].quantile(0.95)

df_europe["media_attention_score"] = (
    np.log1p(df_europe["num_mentions"])
    / np.log1p(mentions_reference)
    * 100
).clip(0, 100)

In [63]:
print("95th percentile mentions:", mentions_reference)

df_europe[
    [
        "num_mentions",
        "media_attention_score"
    ]
].sort_values(
    "num_mentions",
    ascending=False
).head(10)

95th percentile mentions: 10.0


,num_mentions,media_attention_score
342,20,100.0
163,20,100.0
45,20,100.0
930,20,100.0
427,18,100.0
638,18,100.0
509,18,100.0
629,18,100.0
391,14,100.0
885,13,100.0


In [64]:
df_europe["negative_tone_score"] = (
    (-df_europe["avg_tone"]).clip(0, 10)
    / 10
    * 100
)

In [65]:
reference_date = df_europe["event_date"].max()

df_europe["days_old"] = (
    reference_date - df_europe["event_date"]
).dt.days

In [66]:
df_europe["recency_score"] = (
    np.exp(
        -df_europe["days_old"] / 14
    )
    * 100
)

In [67]:
df_europe["attention_score"] = (
    0.55 * df_europe["event_severity_score"] +
    0.20 * df_europe["media_attention_score"] +
    0.15 * df_europe["negative_tone_score"] +
    0.10 * df_europe["recency_score"]
).round(2)

In [68]:
def attention_band(score):

    if score >= 75:
        return "Critical"

    elif score >= 55:
        return "High"

    elif score >= 35:
        return "Medium"

    else:
        return "Low"


df_europe["attention_band"] = (
    df_europe["attention_score"]
    .apply(attention_band)
)

In [69]:
df_europe["attention_score"].describe()

count    462.000000
mean      39.513745
std       15.811965
min       19.820000
25%       28.497500
50%       35.650000
75%       44.765000
max       93.310000
Name: attention_score, dtype: float64

In [70]:
attention_score_distribution = (
    df_europe["attention_band"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"],
        fill_value=0
    )
)

attention_score_distribution

attention_band
Critical     30
High         38
Medium      175
Low         219
Name: count, dtype: int64

In [71]:
top_attention_events = (
    df_europe[
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "quad_class_label",
            "goldstein_scale",
            "num_mentions",
            "avg_tone",
            "location",
            "attention_score",
            "attention_band",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .head(20)
)

top_attention_events

,event_date,actor1,actor2,event_root_label,quad_class_label,goldstein_scale,num_mentions,avg_tone,location,attention_score,attention_band,source_url
396,2026-08-25,GERMANY,NaN,Fight,Material Conflict,-10.0,10,-6.637168,"Berlin, Berlin, Germany",93.31,Critical,https://www.freemalaysiatoday.com/category/wor...
499,2026-08-25,GLASGOW,NaN,Assault,Material Conflict,-9.0,10,-7.002188,"Glasgow, Glasgow City, United Kingdom",91.10,Critical,https://www.glasgowtimes.co.uk/news/26492585.1...
552,2026-08-25,UNITED KINGDOM,NaN,Fight,Material Conflict,-10.0,4,-8.695652,"Hampshire, Hampshire, United Kingdom",89.82,Critical,https://wcbm.com/national-headline/did-the-uk-...
1026,2026-08-25,RUSSIAN,MAYOR,Fight,Material Conflict,-10.0,4,-7.667732,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1111,2026-08-25,UKRAINE,NATIONAL POLICE,Fight,Material Conflict,-10.0,4,-7.667732,"Rostov, Rostovskaya Oblast', Russia",88.28,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1124,2026-08-25,UKRAINE,MAYOR,Fight,Material Conflict,-10.0,4,-7.667732,"Rostov, Rostovskaya Oblast', Russia",88.28,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1040,2026-08-25,RUSSIAN,UKRAINE,Assault,Material Conflict,-9.5,5,-7.667732,"Rostov, Rostovskaya Oblast', Russia",87.60,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1010,2026-08-25,RUSSIAN,NaN,Assault,Material Conflict,-10.0,5,-6.532663,"Moscow, Moskva, Russia",86.44,Critical,https://www.lbc.co.uk/article/uk-faces-growing...
1131,2026-08-25,UKRAINE,RUSSIAN,Fight,Material Conflict,-10.0,3,-7.667732,"Rostov, Rostovskaya Oblast', Russia",86.41,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1041,2026-08-25,RUSSIAN,KHARKIV,Fight,Material Conflict,-10.0,3,-7.667732,"Rostov, Rostovskaya Oblast', Russia",86.41,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...


In [72]:
print(
    "Top 20 events:",
    len(top_attention_events)
)

print(
    "Unique source URLs:",
    top_attention_events["source_url"].nunique()
)

Top 20 events: 20
Unique source URLs: 9


In [73]:
source_frequency = (
    df_europe
    .groupby("source_url")
    .agg(
        event_count=("event_id", "count"),
        max_attention_score=("attention_score", "max"),
        avg_attention_score=("attention_score", "mean")
    )
    .sort_values(
        "event_count",
        ascending=False
    )
)

source_frequency.head(15)

,event_count,max_attention_score,avg_attention_score
source_url,,,
https://www.lbc.co.uk/article/burnham-to-lobby-trump-over-ukraine-defence-plans-as-he-pencils-in-us-trip-for-s-5HjdgTM_2/,27,51.60,30.811481
https://www.lbc.co.uk/article/kremlin-adviser-suggest-uk-factories-could-face-attack-from-unknown-sources-5HjdgTK_2/,19,80.42,48.032105
https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/,18,86.44,49.920556
https://www.thehindubusinessline.com/companies/avaada-inks-mou-with-haryana-for-10000-crore-zero-cbam-green-industrial-campus/article71388104.ece,18,24.11,21.105556
https://igamingbusiness.com/people/industry-tribute-paul-gauselmann-merkur-group/,17,42.65,27.364118
https://www.romania-insider.com/cristian-mungiu-fjord-france-viewers-august-2026,15,34.48,25.177333
https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,14,88.28,83.477143
https://www.romania-insider.com/epp-renew-pnrr-funds-romania-integrity-law-2026,14,51.76,38.468571
https://www.ibtimes.com.au/prince-william-blindsided-harry-meghan-uk-return-1874538,13,45.70,33.905385


In [74]:
top_unique_articles = (
    df_europe
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    [
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "goldstein_scale",
            "location",
            "attention_score",
            "attention_band",
            "source_url"
        ]
    ]
    .head(20)
)

top_unique_articles

,event_date,actor1,actor2,event_root_label,goldstein_scale,location,attention_score,attention_band,source_url
396,2026-08-25,GERMANY,NaN,Fight,-10.0,"Berlin, Berlin, Germany",93.31,Critical,https://www.freemalaysiatoday.com/category/wor...
499,2026-08-25,GLASGOW,NaN,Assault,-9.0,"Glasgow, Glasgow City, United Kingdom",91.10,Critical,https://www.glasgowtimes.co.uk/news/26492585.1...
552,2026-08-25,UNITED KINGDOM,NaN,Fight,-10.0,"Hampshire, Hampshire, United Kingdom",89.82,Critical,https://wcbm.com/national-headline/did-the-uk-...
1026,2026-08-25,RUSSIAN,MAYOR,Fight,-10.0,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,Critical,https://aa.com.tr/en/russia-ukraine-war/at-lea...
1010,2026-08-25,RUSSIAN,NaN,Assault,-10.0,"Moscow, Moskva, Russia",86.44,Critical,https://www.lbc.co.uk/article/uk-faces-growing...
500,2026-08-25,SCOTLAND,NaN,Fight,-10.0,"West Yorkshire, United Kingdom (general), Unit...",83.80,Critical,https://www.mirror.co.uk/news/uk-news/minnie-m...
1130,2026-08-25,KIEV,ROSTOV,Fight,-10.0,"Nizhnekamsk, Tatarstan, Russia",82.83,Critical,http://www.russiaherald.com/news/279263945/ukr...
911,2026-08-25,MACEDONIAN,NaN,Coerce,-5.0,Macedonia,80.74,Critical,https://www.mentalfloss.com/history/medieval-c...
1260,2026-08-25,VENEZUELA,GREENLAND,Fight,-10.0,Venezuela,80.69,Critical,https://www.salon.com/2026/08/25/mark-carney-t...
544,2026-08-25,BRITAIN,UKRAINE,Fight,-10.0,United Kingdom,80.42,Critical,https://www.lbc.co.uk/article/kremlin-adviser-...


In [75]:
lines = response.text.strip().splitlines()

gkg_line = lines[2]

gkg_url = gkg_line.split()[-1]

gkg_url = gkg_url.replace(
    "http://",
    "https://"
)

print("Latest GDELT GKG file:")
print(gkg_url)

Latest GDELT GKG file:
https://data.gdeltproject.org/gdeltv2/20260825114500.gkg.csv.zip


In [76]:
gkg_file_name = gkg_url.split("/")[-1]

gkg_zip_path = (
    raw_dir
    / gkg_file_name
)

gkg_response = requests.get(
    gkg_url,
    timeout=60
)

gkg_response.raise_for_status()

with open(
    gkg_zip_path,
    "wb"
) as file:
    
    file.write(
        gkg_response.content
    )

print("Downloaded:")
print(gkg_zip_path)

print(
    "File size:",
    round(
        gkg_zip_path.stat().st_size
        / 1024,
        2
    ),
    "KB"
)

Downloaded:
c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw\20260825114500.gkg.csv.zip
File size: 5275.56 KB


## 13. Load GDELT GKG

In [77]:
GKG_COLUMNS = [
    "GKGRECORDID",
    "V2DATE",
    "V2SOURCECOLLECTIONIDENTIFIER",
    "V2SOURCECOMMONNAME",
    "V2DOCUMENTIDENTIFIER",
    "V1COUNTS",
    "V2COUNTS",
    "V1THEMES",
    "V2ENHANCEDTHEMES",
    "V1LOCATIONS",
    "V2ENHANCEDLOCATIONS",
    "V1PERSONS",
    "V2ENHANCEDPERSONS",
    "V1ORGANIZATIONS",
    "V2ENHANCEDORGANIZATIONS",
    "V1TONE",
    "V2ENHANCEDDATES",
    "V2GCAM",
    "V2SHARINGIMAGE",
    "V2RELATEDIMAGES",
    "V2SOCIALIMAGEEMBEDS",
    "V2SOCIALVIDEOEMBEDS",
    "V2QUOTATIONS",
    "V2ALLNAMES",
    "V2AMOUNTS",
    "V2TRANSLATIONINFO",
    "V2EXTRASXML"
]

In [78]:
df_gkg = pd.read_csv(
    gkg_zip_path,
    sep="\t",
    header=None,
    names=GKG_COLUMNS,
    compression="zip",
    low_memory=False
)

print("Rows:", len(df_gkg))
print("Columns:", len(df_gkg.columns))

df_gkg.head()

Rows: 1255
Columns: 27


,GKGRECORDID,V2DATE,V2SOURCECOLLECTIONIDENTIFIER,V2SOURCECOMMONNAME,V2DOCUMENTIDENTIFIER,V1COUNTS,V2COUNTS,V1THEMES,V2ENHANCEDTHEMES,V1LOCATIONS,...,V2GCAM,V2SHARINGIMAGE,V2RELATEDIMAGES,V2SOCIALIMAGEEMBEDS,V2SOCIALVIDEOEMBEDS,V2QUOTATIONS,V2ALLNAMES,V2AMOUNTS,V2TRANSLATIONINFO,V2EXTRASXML
0,20260825114500-0,20260825114500,1,tomshardware.com,https://www.tomshardware.com/tech-industry/cyb...,NaN,NaN,TAX_ETHNICITY;TAX_ETHNICITY_CHINESE;TAX_WORLDL...,"TAX_DISEASE_CONVENTIONAL,1003;WB_678_DIGITAL_G...",1#China#CH#CH#35#105#CH,...,"wc:494,c12.1:22,c12.10:36,c12.12:13,c12.13:14,...",https://cdn.mos.cms.futurecdn.net/t6UKEpSvF7JM...,NaN,NaN,https://youtube.com/tomshardware;,NaN,"Matt Callaghan,308;Web Audio,1294;Web Audio,18...","2,suspicious scripts named collina,1333;",NaN,<PAGE_LINKS>https://blog.laserphile.com/2026/0...
1,20260825114500-1,20260825114500,1,tomshardware.com,https://www.tomshardware.com/software/windows/...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_DEVELOPER;TAX_FNCACT_MAN...,"TAX_FNCACT_DEVELOPER,27;TAX_FNCACT_VETERAN,163...",NaN,...,"wc:345,c1.1:1,c12.1:17,c12.10:33,c12.12:11,c12...",https://cdn.mos.cms.futurecdn.net/7ndgbD4kf8Hp...,NaN,NaN,https://youtube.com/tomshardware;,NaN,"Legendary Windows,18;Task Manager,145;Microsof...","11,systems though,295;1.1,on all supported pla...",NaN,<PAGE_LINKS>https://tmog.org/;https://www.toms...
2,20260825114500-2,20260825114500,1,middleeastmonitor.com,https://www.middleeastmonitor.com/20260825-blo...,KILL#24##1#Egypt#EG#EG#27#30#EG;TERROR#24##1#E...,KILL#24##1#Egypt#EG#EG#27#30#EG#580;TERROR#24#...,TAX_FNCACT;TAX_FNCACT_CHILDREN;GEN_HOLIDAY;KIL...,"MANMADE_DISASTER_IMPLIED,640;WB_2203_HUMAN_RIG...","1#Libya#LY#LY#25#17#LY;4#Cairo, Al Qahirah, Eg...",...,"wc:1206,c1.2:6,c1.4:7,c12.1:75,c12.10:110,c12....",https://www.middleeastmonitor.com/wp-content/u...,NaN,NaN,https://youtube.com/@MEMonitor;,NaN,"Sohag Governorate,881;Upper Egypt,896;Dakahlia...","2,children,65;12,of lives,501;2,weeks ago,1008...",NaN,<PAGE_LINKS>https://h1.nu/1ClLK;https://www.mi...
3,20260825114500-3,20260825114500,1,icaew.com,https://www.icaew.com/insights/viewpoints-on-t...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_EMPLOYERS;WB_2690_CATEGO...,"WB_855_LABOR_MARKETS,730;WB_1673_PASSIVE_LABOR...",NaN,...,"wc:402,c12.1:46,c12.10:60,c12.11:1,c12.12:15,c...",https://www.icaew.com/-/media/corporate/images...,NaN,NaN,https://youtube.com/user/icaewnewsroom;,NaN,"Managing Director,507;Neathouse Partners,529;E...",NaN,NaN,<PAGE_LINKS>https://www.icaew.com/technical/bu...
4,20260825114500-4,20260825114500,1,upr.org,https://www.upr.org/npr-news/2026-08-25/spokan...,EVACUATION#9#Mile Nursing Facility#3#Brookdale...,EVACUATION#9#Mile Nursing Facility#3#Brookdale...,CRISISLEX_T01_CAUTION_ADVICE;CRISISLEX_CRISISL...,"MANMADE_DISASTER_IMPLIED,855;MANMADE_DISASTER_...","2#Colorado, United States#US#USCO#39.0646#-105...",...,"wc:1112,c1.3:3,c1.4:2,c12.1:82,c12.10:131,c12....",http://npr-brightspot.s3.amazonaws.com/2f/96/b...,http://npr-brightspot.s3.amazonaws.com/34/ab/1...,NaN,https://youtube.com/channel/UCq_BAhmagzGe08kXU...,NaN,"Sarah Nuss,338;Environmental Protection Agency...","100,of homes,60;3,wildfires,1167;9,Mile Nursin...",NaN,<PAGE_LINKS>https://www.sciencedirect.com/scie...


In [79]:
print(df_gkg.columns.tolist())

['GKGRECORDID', 'V2DATE', 'V2SOURCECOLLECTIONIDENTIFIER', 'V2SOURCECOMMONNAME', 'V2DOCUMENTIDENTIFIER', 'V1COUNTS', 'V2COUNTS', 'V1THEMES', 'V2ENHANCEDTHEMES', 'V1LOCATIONS', 'V2ENHANCEDLOCATIONS', 'V1PERSONS', 'V2ENHANCEDPERSONS', 'V1ORGANIZATIONS', 'V2ENHANCEDORGANIZATIONS', 'V1TONE', 'V2ENHANCEDDATES', 'V2GCAM', 'V2SHARINGIMAGE', 'V2RELATEDIMAGES', 'V2SOCIALIMAGEEMBEDS', 'V2SOCIALVIDEOEMBEDS', 'V2QUOTATIONS', 'V2ALLNAMES', 'V2AMOUNTS', 'V2TRANSLATIONINFO', 'V2EXTRASXML']


In [80]:
df_gkg_core = df_gkg[
    [
        "GKGRECORDID",
        "V2DATE",
        "V2SOURCECOMMONNAME",
        "V2DOCUMENTIDENTIFIER",
        "V1THEMES",
        "V2ENHANCEDTHEMES",
        "V1LOCATIONS",
        "V1ORGANIZATIONS",
        "V1TONE"
    ]
].copy()

In [81]:
df_gkg_core = df_gkg_core.rename(columns={
    "GKGRECORDID": "gkg_record_id",
    "V2DATE": "gkg_date",
    "V2SOURCECOMMONNAME": "source_name",
    "V2DOCUMENTIDENTIFIER": "document_url",
    "V1THEMES": "themes",
    "V2ENHANCEDTHEMES": "enhanced_themes",
    "V1LOCATIONS": "locations",
    "V1ORGANIZATIONS": "organizations",
    "V1TONE": "tone_raw"
})

In [82]:
df_gkg_core[
    [
        "document_url",
        "source_name",
        "themes",
        "enhanced_themes"
    ]
].head(10)

,document_url,source_name,themes,enhanced_themes
0,https://www.tomshardware.com/tech-industry/cyb...,tomshardware.com,TAX_ETHNICITY;TAX_ETHNICITY_CHINESE;TAX_WORLDL...,"TAX_DISEASE_CONVENTIONAL,1003;WB_678_DIGITAL_G..."
1,https://www.tomshardware.com/software/windows/...,tomshardware.com,TAX_FNCACT;TAX_FNCACT_DEVELOPER;TAX_FNCACT_MAN...,"TAX_FNCACT_DEVELOPER,27;TAX_FNCACT_VETERAN,163..."
2,https://www.middleeastmonitor.com/20260825-blo...,middleeastmonitor.com,TAX_FNCACT;TAX_FNCACT_CHILDREN;GEN_HOLIDAY;KIL...,"MANMADE_DISASTER_IMPLIED,640;WB_2203_HUMAN_RIG..."
3,https://www.icaew.com/insights/viewpoints-on-t...,icaew.com,TAX_FNCACT;TAX_FNCACT_EMPLOYERS;WB_2690_CATEGO...,"WB_855_LABOR_MARKETS,730;WB_1673_PASSIVE_LABOR..."
4,https://www.upr.org/npr-news/2026-08-25/spokan...,upr.org,CRISISLEX_T01_CAUTION_ADVICE;CRISISLEX_CRISISL...,"MANMADE_DISASTER_IMPLIED,855;MANMADE_DISASTER_..."
5,https://www.forbes.com/sites/olliebarder/2026/...,forbes.com,CRISISLEX_CRISISLEXREC;ARMEDCONFLICT;EPU_CATS_...,"TAX_WORLDMAMMALS_BEAR,2080;ARMEDCONFLICT,461;E..."
6,https://khvhradio.iheart.com/content/2026-08-2...,iheart.com,CRISISLEX_O01_WEATHER;NATURAL_DISASTER;NATURAL...,"NATURAL_DISASTER_WILDFIRES,1451;CRISISLEX_T01_..."
7,http://www.indiagazette.com/news/279263699/pro...,indiagazette.com,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_MAHARASH...,"WB_1331_HEALTH_TECHNOLOGIES,3192;WB_1350_PHARM..."
8,https://www.tickerreport.com/banking-finance/1...,tickerreport.com,ECON_STOCKMARKET;TAX_FNCACT;TAX_FNCACT_ANALYST...,"ECON_STOCKMARKET,102;ECON_STOCKMARKET,3327;ECO..."
9,http://www.indiagazette.com/news/279263739/ind...,indiagazette.com,UNGP_FORESTS_RIVERS_OCEANS;EPU_ECONOMY_HISTORI...,"TAX_WORLDLANGUAGES_NAGAR,4094;TAX_ETHNICITY_IN..."


In [83]:
matched_urls = df_europe["source_url"].isin(
    df_gkg_core["document_url"]
)

print("European events:", len(df_europe))
print("Events with GKG match:", matched_urls.sum())

print(
    "Match rate:",
    round(
        matched_urls.mean() * 100,
        2
    ),
    "%"
)

European events: 462
Events with GKG match: 462
Match rate: 100.0 %


## 14. Validate GKG Join

In [84]:
print("GKG rows:", len(df_gkg_core))

print(
    "Unique document URLs:",
    df_gkg_core["document_url"].nunique()
)

print(
    "Duplicated document URLs:",
    df_gkg_core["document_url"].duplicated().sum()
)

GKG rows: 1255
Unique document URLs: 1255
Duplicated document URLs: 0


In [85]:
df_security = df_europe.merge(
    df_gkg_core,
    left_on="source_url",
    right_on="document_url",
    how="left",
    validate="many_to_one"
)

In [86]:
print("Before merge:", len(df_europe))
print("After merge:", len(df_security))

print(
    "Missing GKG themes:",
    df_security["themes"].isna().sum()
)

Before merge: 462
After merge: 462
Missing GKG themes: 3


## 15. GKG Theme Inspection

In [87]:
df_articles = (
    df_gkg_core[
        df_gkg_core["document_url"].isin(
            df_europe["source_url"]
        )
    ]
    .copy()
)

print("European / strategic events:", len(df_europe))
print("Unique related articles:", len(df_articles))

European / strategic events: 462
Unique related articles: 105


In [88]:
def split_gkg_themes(value):
    """
    Converts the semicolon-separated GKG theme field
    into a Python list.
    """

    if pd.isna(value):
        return []

    return [
        theme.strip()
        for theme in str(value).split(";")
        if theme.strip()
    ]


df_articles["theme_list"] = (
    df_articles["themes"]
    .apply(split_gkg_themes)
)

In [89]:
df_articles[
    [
        "source_name",
        "document_url",
        "theme_list"
    ]
].head(10)

,source_name,document_url,theme_list
12,lbc.co.uk,https://www.lbc.co.uk/article/man-charged-pens...,"[KILL, CRISISLEX_T03_DEAD, CRIME_COMMON_ROBBER..."
40,romania-insider.com,https://www.romania-insider.com/mirabela-gradi...,"[TAX_ETHNICITY, TAX_ETHNICITY_ROMANIAN, TAX_WO..."
42,athens-times.com,https://athens-times.com/papastaurou-sustainab...,"[TAX_ETHNICITY, TAX_ETHNICITY_GREEK, TAX_WORLD..."
47,irishnews.com,https://www.irishnews.com/news/northern-irelan...,"[TAX_FNCACT, TAX_FNCACT_MINISTER, LEGISLATION,..."
49,express.co.uk,https://www.express.co.uk/news/uk/2242229/stra...,"[TAX_FNCACT, TAX_FNCACT_MAN, KILL, CRISISLEX_T..."
51,cphpost.dk,https://cphpost.dk/2026-08-25/business-educati...,"[TAX_FNCACT, TAX_FNCACT_AUTHOR, TAX_FNCACT_DIR..."
76,glasgowtimes.co.uk,https://www.glasgowtimes.co.uk/news/scottish-n...,"[MARITIME_INCIDENT, MARITIME, MANMADE_DISASTER..."
86,yahoo.com,https://finance.yahoo.com/small-business/artic...,"[TAX_FNCACT, TAX_FNCACT_FOUNDER, TAX_FNCACT_SP..."
98,wallpaper.com,https://www.wallpaper.com/design-interiors/vis...,"[TAX_ETHNICITY, TAX_ETHNICITY_ITALIAN, TAX_WOR..."
102,express.co.uk,https://www.express.co.uk/news/politics/224224...,"[EPU_POLICY, EPU_POLICY_REFORM, TAX_FNCACT, TA..."


In [90]:
theme_frequency = (
    df_articles["theme_list"]
    .explode()
    .dropna()
    .value_counts()
    .reset_index()
)

theme_frequency.columns = [
    "theme",
    "article_count"
]

theme_frequency.head(50)

,theme,article_count
0,TAX_FNCACT,99
1,TAX_ETHNICITY,70
2,EPU_POLICY,59
3,TAX_WORLDLANGUAGES,57
4,CRISISLEX_CRISISLEXREC,51
5,UNGP_FORESTS_RIVERS_OCEANS,48
6,CRISISLEX_C07_SAFETY,40
7,USPEC_POLICY1,37
8,EPU_ECONOMY_HISTORIC,37
9,WB_696_PUBLIC_SECTOR_MANAGEMENT,35


In [91]:
sample_articles = (
    df_security[
        [
            "source_name",
            "document_url",
            "event_root_label",
            "location",
            "attention_score",
            "themes"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="document_url"
    )
    .head(15)
)

sample_articles

,source_name,document_url,event_root_label,location,attention_score,themes
96,freemalaysiatoday.com,https://www.freemalaysiatoday.com/category/wor...,Fight,"Berlin, Berlin, Germany",93.31,TAX_ETHNICITY;TAX_ETHNICITY_GERMAN;TAX_WORLDLA...
180,glasgowtimes.co.uk,https://www.glasgowtimes.co.uk/news/26492585.1...,Assault,"Glasgow, Glasgow City, United Kingdom",91.10,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE...
233,wcbm.com,https://wcbm.com/national-headline/did-the-uk-...,Fight,"Hampshire, Hampshire, United Kingdom",89.82,KILL;HATE_SPEECH;DISCRIMINATION;SECURITY_SERVI...
370,aa.com.tr,https://aa.com.tr/en/russia-ukraine-war/at-lea...,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,KILL;CRISISLEX_T03_DEAD;WOUND;CRISISLEX_CRISIS...
354,lbc.co.uk,https://www.lbc.co.uk/article/uk-faces-growing...,Assault,"Moscow, Moskva, Russia",86.44,TAX_ETHNICITY;TAX_ETHNICITY_RUSSIAN;TAX_WORLDL...
181,mirror.co.uk,https://www.mirror.co.uk/news/uk-news/minnie-m...,Fight,"West Yorkshire, United Kingdom (general), Unit...",83.80,CRISISLEX_CRISISLEXREC;KILL;TAX_FNCACT;TAX_FNC...
441,russiaherald.com,http://www.russiaherald.com/news/279263945/ukr...,Fight,"Nizhnekamsk, Tatarstan, Russia",82.83,EPU_CATS_MIGRATION_FEAR_FEAR;EPU_CATS_NATIONAL...
325,mentalfloss.com,https://www.mentalfloss.com/history/medieval-c...,Coerce,Macedonia,80.74,TAX_WORLDMAMMALS;TAX_WORLDMAMMALS_BEAR;TAX_FNC...
459,salon.com,https://www.salon.com/2026/08/25/mark-carney-t...,Fight,Venezuela,80.69,ARMEDCONFLICT;EPU_CATS_NATIONAL_SECURITY;WB_24...
225,lbc.co.uk,https://www.lbc.co.uk/article/kremlin-adviser-...,Fight,United Kingdom,80.42,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_RUSSIA;T...


## 16. Security Theme Discovery

In [92]:
security_search_terms = [
    "ARMED",
    "MILITARY",
    "DEFEN",
    "WEAPON",
    "MISSILE",
    "WAR",
    "CONFLICT",
    "CYBER",
    "HACK",
    "SANCTION",
    "ENERGY",
    "OIL",
    "GAS",
    "NUCLEAR",
    "TERROR",
    "SECURITY",
    "NATO",
    "RUSSIA",
    "UKRAINE"
]

In [93]:
security_theme_candidates = theme_frequency[
    theme_frequency["theme"]
    .str.contains(
        "|".join(security_search_terms),
        case=False,
        na=False
    )
].copy()

security_theme_candidates.head(100)

,theme,article_count
26,WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE,21
27,SECURITY_SERVICES,20
34,ARMEDCONFLICT,18
35,EPU_CATS_NATIONAL_SECURITY,18
41,TAX_MILITARY_TITLE,16
...,...,...
787,WB_532_BIOFUELS_ENERGY,1
849,WB_739_POLITICAL_VIOLENCE_AND_CIVIL_WAR,1
871,TAX_WEAPONS_EXPLOSIVES,1
876,TAX_WEAPONS_AIR_DEFENSE_SYSTEM,1


In [94]:
def find_themes(term):
    return theme_frequency[
        theme_frequency["theme"]
        .str.contains(
            term,
            case=False,
            na=False
        )
    ]

In [95]:
find_themes("ARMED")

,theme,article_count
34,ARMEDCONFLICT,18


In [96]:
find_themes("MILITARY")

,theme,article_count
41,TAX_MILITARY_TITLE,16
96,TAX_MILITARY_TITLE_OFFICERS,8
118,TAX_MILITARY_TITLE_OFFICER,7
144,MILITARY,6
327,TAX_MILITARY_TITLE_SUPERINTENDENT,2
364,TAX_MILITARY_TITLE_COMMANDER,2
523,TAX_MILITARY_TITLE_SOLDIERS,1
527,SLFID_MILITARY_READINESS,1
528,SLFID_MILITARY_SPENDING,1
533,TAX_MILITARY_TITLE_AIR_CHIEF_MARSHAL,1


In [97]:
find_themes("CYBER")

,theme,article_count
171,CYBER_ATTACK,5


In [98]:
find_themes("SANCTION")

,theme,article_count
350,SANCTIONS,2


In [99]:
find_themes("ENERGY")

,theme,article_count
44,WB_507_ENERGY_AND_EXTRACTIVES,16
162,WB_525_RENEWABLE_ENERGY,5
474,WB_529_WIND_ENERGY,1
715,WB_528_SOLAR_ENERGY,1
718,WB_533_ENERGY_EFFICIENCY,1
787,WB_532_BIOFUELS_ENERGY,1


In [100]:
find_themes("CONFLICT")

,theme,article_count
26,WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE,21
34,ARMEDCONFLICT,18
66,WB_2433_CONFLICT_AND_VIOLENCE,13
94,WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT,9


In [101]:
pd.set_option("display.max_colwidth", None)

In [102]:
security_samples = (
    df_security
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="document_url"
    )
    [
        [
            "document_url",
            "event_root_label",
            "location",
            "attention_score",
            "themes"
        ]
    ]
    .head(10)
)

security_samples

,document_url,event_root_label,location,attention_score,themes
96,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz,Fight,"Berlin, Berlin, Germany",93.31,TAX_ETHNICITY;TAX_ETHNICITY_GERMAN;TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_GERMAN;GENERAL_GOVERNMENT;EPU_POLICY;EPU_POLICY_GOVERNMENT;DRONES;CRISISLEX_CRISISLEXREC;SOC_POINTSOFINTEREST;SOC_POINTSOFINTEREST_AIRPORT;WB_135_TRANSPORT;WB_1803_TRANSPORT_INFRASTRUCTURE;WB_1804_AIRPORTS;TAX_FNCACT;TAX_FNCACT_CHANCELLOR;ALLIANCE;CRISISLEX_C07_SAFETY;TAX_FNCACT_AUTHORITIES;EPU_POLICY_AUTHORITIES;WB_678_DIGITAL_GOVERNMENT;WB_694_BROADCAST_AND_MEDIA;WB_133_INFORMATION_AND_COMMUNICATION_TECHNOLOGIES;TAX_ETHNICITY_RUSSIAN;TAX_WORLDLANGUAGES_RUSSIAN;TAX_ETHNICITY_UKRAINIAN;TAX_WORLDLANGUAGES_UKRAINIAN;MANMADE_DISASTER_IMPLIED;TAX_WEAPONS;TAX_WEAPONS_EXPLOSIVES;MEDIA_MSM;TRIAL;TAX_FNCACT_PROSECUTOR;ACT_MAKESTATEMENT;TAX_WORLDLANGUAGES_RUSSIA;
180,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/,Assault,"Glasgow, Glasgow City, United Kingdom",91.10,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE;CRISISLEX_C07_SAFETY;TAX_ETHNICITY;TAX_ETHNICITY_SCOTTISH;GENERAL_GOVERNMENT;EPU_POLICY;EPU_POLICY_GOVERNMENT;RAPE;UNGP_CRIME_VIOLENCE;WB_2433_CONFLICT_AND_VIOLENCE;WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE;GENDER_VIOLENCE;WB_742_YOUTH_AND_GENDER_BASED_VIOLENCE;WB_2441_DOMESTIC_VIOLENCE;WB_738_SOCIAL_COHESION;WB_134_SOCIAL_DEVELOPMENT;EPU_CATS_MIGRATION_FEAR_FEAR;SOC_GENERALCRIME;TAX_FNCACT_WOMEN;UNGP_GENDER_EQUALITY;WB_696_PUBLIC_SECTOR_MANAGEMENT;WB_840_JUSTICE;TAX_FNCACT_SECRETARY;TAX_FNCACT_VICTIMS;CRISISLEX_CRISISLEXREC;CRISISLEX_T08_MISSINGFOUNDTRAPPEDPEOPLE;CRISISLEX_T03_DEAD;LEGISLATION;USPEC_POLITICS_GENERAL1;USPEC_POLICY1;EPU_POLICY_LEGISLATION;TRIAL;TAX_FNCACT_PROSECUTORS;WB_615_GENDER;WB_695_POVERTY;INEQUALITY;HARASSMENT;TAX_FNCACT_MINISTER;AFFECT;CRISISLEX_T02_INJURED;WB_2443_RAPE_AND_SEXUAL_VIOLENCE;WB_2024_ANTI_CORRUPTION_AUTHORITIES;WB_831_GOVERNANCE;WB_832_ANTI_CORRUPTION;WB_2026_PREVENTION;WB_930_VIOLENCE_PREVENTION;
233,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/,Fight,"Hampshire, Hampshire, United Kingdom",89.82,KILL;HATE_SPEECH;DISCRIMINATION;SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE;CRISISLEX_C07_SAFETY;WB_1428_INJURY;WB_1406_DISEASES;WB_621_HEALTH_NUTRITION_AND_POPULATION;WB_1427_NON_COMMUNICABLE_DISEASE_AND_INJURY;CRISISLEX_CRISISLEXREC;CRISISLEX_T02_INJURED;WB_2024_ANTI_CORRUPTION_AUTHORITIES;WB_696_PUBLIC_SECTOR_MANAGEMENT;WB_840_JUSTICE;WB_2025_INVESTIGATION;WB_831_GOVERNANCE;WB_832_ANTI_CORRUPTION;WB_1014_CRIMINAL_JUSTICE;DISCRIMINATION_RACE;DISCRIMINATION_RACE_RACIST;DISCRIMINATION_RACE_RACISM;UNGP_FREEDOM_FROM_DISCRIMINATION;SOC_GENERALCRIME;TRIAL;USPEC_POLICY1;EPU_POLICY;EPU_POLICY_POLICY;CRISISLEX_T11_UPDATESSYMPATHY;EPU_CATS_MIGRATION_FEAR_FEAR;EPU_POLICY_POLITICAL;TAX_FNCACT_CLOWNS;TAX_FNCACT_CLOWN;TAX_RELIGION;TAX_RELIGION_JEWISH;TAX_ETHNICITY;TAX_ETHNICITY_JEWISH;TAX_MILITARY_TITLE;TAX_MILITARY_TITLE_OFFICER;TAX_FNCACT_OFFICER;REL_ANTISEMITISM;TAX_FNCACT_LEADER;ARREST;USPEC_UNCERTAINTY1;WB_1160_SHOCKS_AND_VULNERABILITY;WB_695_POVERTY;
370,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,KILL;CRISISLEX_T03_DEAD;WOUND;CRISISLEX_CRISISLEXREC;CRISISLEX_C03_WELLBEING_HEALTH;CRISISLEX_T02_INJURED;TAX_ETHNICITY;TAX_ETHNICITY_RUSSIAN;TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_RUSSIAN;TAX_ETHNICITY_UKRAINIAN;TAX_WORLDLANGUAGES_UKRAINIAN;TAX_FNCACT;TAX_FNCACT_OFFICIALS;BORDER;TAX_FNCACT_GUARD;DRONES;CRISISLEX_C04_LOGISTICS_TRANSPORT;MANMADE_DISASTER_IMPLIED;SEIGE;CHECKPOINT;UNREST_CHECKPOINT;CRISISLEX_O02_RESPONSEAGENCIESATCRISIS;DISASTER_FIRE;CRISISLEX_T01_CAUTION_ADVICE;EPU_ECONOMY_HISTORIC;WB_1921_PRIVATE_SECTOR_DEVELOPMENT;WB_1202_INDUSTRIAL_ZONES;WB_862_GROWTH_POLES_AND_E

## 17. Security Relevance Layer

In [103]:
GKG_SECURITY_THEMES = {

    "Defence & Military": {
        "MILITARY",
        "MILITARY_COOPERATION",
        "TAX_WEAPONS_DRONE_STRIKE",
        "TAX_WEAPONS_ARTILLERY",
        "ARMEDCONFLICT",
        "WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT"
    },

    "Cybersecurity": {
        "CYBER_ATTACK",
        "WB_670_ICT_SECURITY",
        "TAX_FNCACT_HACKER",
        "TAX_FNCACT_HACKERS"
    },

    "Energy Security": {
        "WB_507_ENERGY_AND_EXTRACTIVES",
        "ENV_OIL",
        "ENV_NATURALGAS",
        "WB_539_OIL_AND_GAS_POLICY_STRATEGY_AND_INSTITUTIONS",
        "WB_2290_OIL_AND_GAS_EXPORT",
        "WB_544_MID_AND_DOWNSTREAM_OIL_AND_GAS",
        "WB_2273_UPSTREAM_OIL_AND_GAS",
        "WB_525_RENEWABLE_ENERGY",
        "WB_532_BIOFUELS_ENERGY",
        "WB_548_PPP_IN_OIL_AND_GAS"
    },

    "Sanctions & Economic Security": {
        "SANCTIONS"
    },

    "Conflict & Geopolitical Tensions": {
        "ARMEDCONFLICT",
        "WB_2433_CONFLICT_AND_VIOLENCE",
        "WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE",
        "WB_739_POLITICAL_VIOLENCE_AND_CIVIL_WAR",
        "WB_2462_POLITICAL_VIOLENCE_AND_WAR",
        "TERROR"
    }
}

In [104]:
def classify_gkg_security_domains(theme_list):
    """
    Classifies an article into one or more security domains
    using high-confidence GDELT GKG themes.
    """

    if not isinstance(theme_list, list):
        return []

    detected_domains = []

    theme_set = set(theme_list)

    for domain, relevant_themes in GKG_SECURITY_THEMES.items():

        if theme_set.intersection(relevant_themes):
            detected_domains.append(domain)

    return detected_domains

In [105]:
df_security["theme_list"] = (
    df_security["themes"]
    .apply(split_gkg_themes)
)

In [106]:
df_security["security_domains"] = (
    df_security["theme_list"]
    .apply(classify_gkg_security_domains)
)

In [107]:
df_security["security_relevant"] = (
    df_security["security_domains"]
    .apply(lambda x: len(x) > 0)
)

In [108]:
df_relevant = df_security[
    df_security["security_relevant"]
].copy()

df_not_relevant = df_security[
    ~df_security["security_relevant"]
].copy()

In [109]:
print("European / strategic events:", len(df_security))

print(
    "Security relevant events:",
    len(df_relevant)
)

print(
    "Excluded events:",
    len(df_not_relevant)
)

print(
    "Security relevance rate:",
    round(
        len(df_relevant)
        / len(df_security)
        * 100,
        2
    ),
    "%"
)

European / strategic events: 462
Security relevant events: 261
Excluded events: 201
Security relevance rate: 56.49 %


In [110]:
domain_distribution = (
    df_relevant["security_domains"]
    .explode()
    .value_counts()
    .reset_index()
)

domain_distribution.columns = [
    "security_domain",
    "events"
]

domain_distribution

,security_domain,events
0,Conflict & Geopolitical Tensions,197
1,Defence & Military,164
2,Energy Security,107
3,Cybersecurity,54
4,Sanctions & Economic Security,11


In [111]:
relevance_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "location",
            "attention_score",
            "security_domains",
            "security_relevant",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_check

,event_date,actor1,actor2,event_root_label,location,attention_score,security_domains,security_relevant,source_url
96,2026-08-25,GERMANY,NaN,Fight,"Berlin, Berlin, Germany",93.31,[],False,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz
180,2026-08-25,GLASGOW,NaN,Assault,"Glasgow, Glasgow City, United Kingdom",91.10,[Conflict & Geopolitical Tensions],True,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/
233,2026-08-25,UNITED KINGDOM,NaN,Fight,"Hampshire, Hampshire, United Kingdom",89.82,[],False,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/
370,2026-08-25,RUSSIAN,MAYOR,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407
354,2026-08-25,RUSSIAN,NaN,Assault,"Moscow, Moskva, Russia",86.44,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/
181,2026-08-25,SCOTLAND,NaN,Fight,"West Yorkshire, United Kingdom (general), United Kingdom",83.80,[],False,https://www.mirror.co.uk/news/uk-news/minnie-merriman-arbroath-murdered-mourners-37589650
441,2026-08-25,KIEV,ROSTOV,Fight,"Nizhnekamsk, Tatarstan, Russia",82.83,[],False,http://www.russiaherald.com/news/279263945/ukraine-a-western-backed-zelensky-concentration-camp-moscow
325,2026-08-25,MACEDONIAN,NaN,Coerce,Macedonia,80.74,[],False,https://www.mentalfloss.com/history/medieval-court-cases-animals-trial
459,2026-08-25,VENEZUELA,GREENLAND,Fight,Venezuela,80.69,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://www.salon.com/2026/08/25/mark-carney-tells-trump-he-will-never-own-canada-amid-tariffs/
225,2026-08-25,BRITAIN,UKRAINE,Fight,United Kingdom,80.42,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://www.lbc.co.uk/article/kremlin-adviser-suggest-uk-factories-could-face-attack-from-unknown-sources-5HjdgTK_2/


## 18. Event Actor Context

In [112]:
actor_context = df_gdelt[
    [
        "GLOBALEVENTID",
        "Actor1Type1Code",
        "Actor2Type1Code",
        "Actor1KnownGroupCode",
        "Actor2KnownGroupCode"
    ]
].copy()

actor_context = actor_context.rename(columns={
    "GLOBALEVENTID": "event_id",
    "Actor1Type1Code": "actor1_type",
    "Actor2Type1Code": "actor2_type",
    "Actor1KnownGroupCode": "actor1_group",
    "Actor2KnownGroupCode": "actor2_group"
})

In [113]:
print(
    "Rows:",
    len(actor_context)
)

print(
    "Unique event IDs:",
    actor_context["event_id"].nunique()
)

Rows: 1284
Unique event IDs: 1284


In [114]:
df_security = df_security.merge(
    actor_context,
    on="event_id",
    how="left",
    validate="one_to_one"
)

In [115]:
print("Rows after actor merge:", len(df_security))

Rows after actor merge: 462


In [116]:
actor_relevance_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor1_group",
            "actor2",
            "actor2_country",
            "actor2_type",
            "actor2_group",
            "event_root_label",
            "location",
            "attention_score",
            "security_domains",
            "security_relevant",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

actor_relevance_check

,event_date,actor1,actor1_country,actor1_type,actor1_group,actor2,actor2_country,actor2_type,actor2_group,event_root_label,location,attention_score,security_domains,security_relevant,source_url
96,2026-08-25,GERMANY,DEU,NaN,NaN,NaN,NaN,NaN,NaN,Fight,"Berlin, Berlin, Germany",93.31,[],False,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz
180,2026-08-25,GLASGOW,GBR,NaN,NaN,NaN,NaN,NaN,NaN,Assault,"Glasgow, Glasgow City, United Kingdom",91.10,[Conflict & Geopolitical Tensions],True,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/
233,2026-08-25,UNITED KINGDOM,GBR,COP,NaN,NaN,NaN,NaN,NaN,Fight,"Hampshire, Hampshire, United Kingdom",89.82,[],False,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/
370,2026-08-25,RUSSIAN,RUS,NaN,NaN,MAYOR,NaN,GOV,NaN,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407
354,2026-08-25,RUSSIAN,RUS,NaN,NaN,NaN,NaN,NaN,NaN,Assault,"Moscow, Moskva, Russia",86.44,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/
181,2026-08-25,SCOTLAND,GBR,NaN,NaN,NaN,NaN,NaN,NaN,Fight,"West Yorkshire, United Kingdom (general), United Kingdom",83.80,[],False,https://www.mirror.co.uk/news/uk-news/minnie-merriman-arbroath-murdered-mourners-37589650
441,2026-08-25,KIEV,UKR,NaN,NaN,ROSTOV,RUS,NaN,NaN,Fight,"Nizhnekamsk, Tatarstan, Russia",82.83,[],False,http://www.russiaherald.com/news/279263945/ukraine-a-western-backed-zelensky-concentration-camp-moscow
325,2026-08-25,MACEDONIAN,MKD,NaN,NaN,NaN,NaN,NaN,NaN,Coerce,Macedonia,80.74,[],False,https://www.mentalfloss.com/history/medieval-court-cases-animals-trial
459,2026-08-25,VENEZUELA,VEN,NaN,NaN,GREENLAND,DNK,NaN,NaN,Fight,Venezuela,80.69,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://www.salon.com/2026/08/25/mark-carney-tells-trump-he-will-never-own-canada-amid-tariffs/
225,2026-08-25,BRITAIN,GBR,NaN,NaN,UKRAINE,UKR,NaN,NaN,Fight,United Kingdom,80.42,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://www.lbc.co.uk/article/kremlin-adviser-suggest-uk-factories-could-face-attack-from-unknown-sources-5HjdgTK_2/


In [117]:
print("Actor 1 types:")

print(
    df_security["actor1_type"]
    .value_counts(dropna=False)
    .head(30)
)

Actor 1 types:
actor1_type
NaN    300
GOV     65
BUS     19
COP     14
EDU     13
MED     11
MIL      9
CVL      5
IGO      5
MNC      5
ELI      4
LEG      4
SPY      3
HLH      2
CRM      1
LAB      1
UAF      1
Name: count, dtype: int64


In [118]:
print("Actor 2 types:")

print(
    df_security["actor2_type"]
    .value_counts(dropna=False)
    .head(30)
)

Actor 2 types:
actor2_type
NaN    330
GOV     51
EDU     14
BUS     10
LEG      8
CVL      8
COP      7
MIL      7
MED      6
SPY      5
MNC      4
ELI      4
IGO      3
HLH      2
CRM      1
LAB      1
UAF      1
Name: count, dtype: int64


In [119]:
STRATEGIC_ACTOR_TYPES = {
    "GOV",  # Government
    "MIL",  # Military
    "REB",  # Rebels
    "INS",  # Insurgents
    "SEP",  # Separatists
    "SPY",  # Intelligence services
    "UAF",  # Unaligned armed forces
    "IGO"   # International governmental organisations
}

In [120]:
df_security["strategic_actor"] = (
    df_security["actor1_type"].isin(STRATEGIC_ACTOR_TYPES)
    |
    df_security["actor2_type"].isin(STRATEGIC_ACTOR_TYPES)
)

In [121]:
print(
    "Events with strategic actor:",
    df_security["strategic_actor"].sum()
)

Events with strategic actor: 140


In [122]:
COUNTRY_TO_CAMEO = {
    "Albania": "ALB",
    "Austria": "AUT",
    "Belarus": "BLR",
    "Belgium": "BEL",
    "Bosnia and Herzegovina": "BIH",
    "Bulgaria": "BGR",
    "Croatia": "HRV",
    "Cyprus": "CYP",
    "Czechia": "CZE",
    "Denmark": "DNK",
    "Estonia": "EST",
    "Finland": "FIN",
    "France": "FRA",
    "Germany": "DEU",
    "Greece": "GRC",
    "Hungary": "HUN",
    "Iceland": "ISL",
    "Ireland": "IRL",
    "Italy": "ITA",
    "Latvia": "LVA",
    "Lithuania": "LTU",
    "Luxembourg": "LUX",
    "Malta": "MLT",
    "Moldova": "MDA",
    "Montenegro": "MNE",
    "Netherlands": "NLD",
    "North Macedonia": "MKD",
    "Norway": "NOR",
    "Poland": "POL",
    "Portugal": "PRT",
    "Romania": "ROU",
    "Serbia": "SRB",
    "Slovakia": "SVK",
    "Slovenia": "SVN",
    "Spain": "ESP",
    "Sweden": "SWE",
    "Switzerland": "CHE",
    "Türkiye": "TUR",
    "Ukraine": "UKR",
    "United Kingdom": "GBR",
    "Russia": "RUS",
    "Georgia": "GEO",
    "Armenia": "ARM",
    "Azerbaijan": "AZE",
    "Monaco": "MCO",
    "Andorra": "AND",
    "Liechtenstein": "LIE",
    "San Marino": "SMR",
    "Vatican City": "VAT"
}

In [123]:
def get_location_cameo(location_countries):
    
    if not isinstance(location_countries, list):
        return None
    
    if len(location_countries) == 0:
        return None
    
    country = location_countries[0]
    
    return COUNTRY_TO_CAMEO.get(country)

In [124]:
df_security["location_cameo"] = (
    df_security["location_countries"]
    .apply(get_location_cameo)
)

In [125]:
def detect_cross_border_context(row):
    
    location_code = row["location_cameo"]
    
    if pd.isna(location_code):
        return False
    
    actor_codes = [
        row["actor1_country"],
        row["actor2_country"]
    ]
    
    actor_codes = [
        code
        for code in actor_codes
        if pd.notna(code)
    ]
    
    for actor_code in actor_codes:
        
        if (
            actor_code in MONITORED_ISO3
            and actor_code != location_code
        ):
            return True
    
    return False

In [126]:
df_security["cross_border_context"] = (
    df_security.apply(
        detect_cross_border_context,
        axis=1
    )
)

In [127]:
df_security["strategic_context"] = (
    df_security["strategic_actor"]
    |
    df_security["cross_border_context"]
)

In [128]:
print(
    "Strategic actor:",
    df_security["strategic_actor"].sum()
)

print(
    "Cross-border context:",
    df_security["cross_border_context"].sum()
)

print(
    "Strategic context total:",
    df_security["strategic_context"].sum()
)

Strategic actor: 140
Cross-border context: 115
Strategic context total: 231


In [129]:
DIRECT_DEFENCE_THEMES = {
    "MILITARY_COOPERATION",
    "TAX_WEAPONS_DRONE_STRIKE",
    "TAX_WEAPONS_ARTILLERY"
}

BROAD_DEFENCE_THEMES = {
    "MILITARY",
    "ARMEDCONFLICT",
    "WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT"
}


CYBER_THEMES = {
    "CYBER_ATTACK",
    "WB_670_ICT_SECURITY",
    "TAX_FNCACT_HACKER",
    "TAX_FNCACT_HACKERS"
}


SANCTIONS_THEMES = {
    "SANCTIONS"
}


DIRECT_CONFLICT_THEMES = {
    "WB_739_POLITICAL_VIOLENCE_AND_CIVIL_WAR",
    "WB_2462_POLITICAL_VIOLENCE_AND_WAR",
    "TERROR"
}

BROAD_CONFLICT_THEMES = {
    "ARMEDCONFLICT",
    "WB_2433_CONFLICT_AND_VIOLENCE",
    "WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE"
}


ENERGY_THEMES = {
    "WB_507_ENERGY_AND_EXTRACTIVES",
    "ENV_OIL",
    "ENV_NATURALGAS",
    "WB_539_OIL_AND_GAS_POLICY_STRATEGY_AND_INSTITUTIONS",
    "WB_2290_OIL_AND_GAS_EXPORT",
    "WB_544_MID_AND_DOWNSTREAM_OIL_AND_GAS",
    "WB_2273_UPSTREAM_OIL_AND_GAS",
    "WB_525_RENEWABLE_ENERGY",
    "WB_532_BIOFUELS_ENERGY",
    "WB_548_PPP_IN_OIL_AND_GAS"
}

SECURITY_CONFLICT_ROOTS = {
    13,  # Threaten
    15,  # Exhibit Force Posture
    16,  # Reduce Relations
    17,  # Coerce
    18,  # Assault
    19,  # Fight
    20   # Unconventional Mass Violence
}

In [130]:
def classify_security_domains_v2(row):
    
    themes = set(row["theme_list"])
    
    domains = []
    
    strategic_context = row["strategic_context"]
    root_code = row["event_root_code"]
    
    
    # -------------------------
    # CYBERSECURITY
    # -------------------------
    
    if themes.intersection(CYBER_THEMES):
        domains.append("Cybersecurity")
    
    
    # -------------------------
    # SANCTIONS
    # -------------------------
    
    if themes.intersection(SANCTIONS_THEMES):
        domains.append(
            "Sanctions & Economic Security"
        )
    
    
    # -------------------------
    # DEFENCE & MILITARY
    # -------------------------
    
    direct_defence = bool(
        themes.intersection(
            DIRECT_DEFENCE_THEMES
        )
    )
    
    contextual_defence = (
        bool(
            themes.intersection(
                BROAD_DEFENCE_THEMES
            )
        )
        and strategic_context
    )
    
    if direct_defence or contextual_defence:
        domains.append(
            "Defence & Military"
        )
    
    
    # -------------------------
    # CONFLICT
    # -------------------------
    
    direct_conflict = bool(
        themes.intersection(
            DIRECT_CONFLICT_THEMES
        )
    )
    
    contextual_conflict = (
        bool(
            themes.intersection(
                BROAD_CONFLICT_THEMES
            )
        )
        and strategic_context
        and root_code in SECURITY_CONFLICT_ROOTS
    )
    
    if direct_conflict or contextual_conflict:
        domains.append(
            "Conflict & Geopolitical Tensions"
        )
    
    
    # -------------------------
    # ENERGY SECURITY
    # -------------------------
    
    energy_theme = bool(
        themes.intersection(
            ENERGY_THEMES
        )
    )
    
    security_context = (
        strategic_context
        or bool(
            themes.intersection(
                SANCTIONS_THEMES
                | BROAD_CONFLICT_THEMES
                | DIRECT_CONFLICT_THEMES
            )
        )
    )
    
    if energy_theme and security_context:
        domains.append(
            "Energy Security"
        )
    
    
    return domains

In [131]:
df_security["security_domains_v2"] = (
    df_security.apply(
        classify_security_domains_v2,
        axis=1
    )
)

df_security["security_relevant_v2"] = (
    df_security["security_domains_v2"]
    .apply(lambda x: len(x) > 0)
)

In [132]:
print(
    "V1 relevant:",
    df_security["security_relevant"].sum()
)

print(
    "V2 relevant:",
    df_security["security_relevant_v2"].sum()
)

V1 relevant: 261
V2 relevant: 168


In [133]:
relevance_v2_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor2",
            "actor2_country",
            "actor2_type",
            "event_root_label",
            "location",
            "strategic_actor",
            "cross_border_context",
            "attention_score",
            "security_domains_v2",
            "security_relevant_v2",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_v2_check

,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_root_label,location,strategic_actor,cross_border_context,attention_score,security_domains_v2,security_relevant_v2,source_url
96,2026-08-25,GERMANY,DEU,NaN,NaN,NaN,NaN,Fight,"Berlin, Berlin, Germany",False,False,93.31,[],False,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz
180,2026-08-25,GLASGOW,GBR,NaN,NaN,NaN,NaN,Assault,"Glasgow, Glasgow City, United Kingdom",False,False,91.10,[],False,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/
233,2026-08-25,UNITED KINGDOM,GBR,COP,NaN,NaN,NaN,Fight,"Hampshire, Hampshire, United Kingdom",False,False,89.82,[],False,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/
370,2026-08-25,RUSSIAN,RUS,NaN,MAYOR,NaN,GOV,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",True,False,88.28,"[Defence & Military, Conflict & Geopolitical Tensions, Energy Security]",True,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407
354,2026-08-25,RUSSIAN,RUS,NaN,NaN,NaN,NaN,Assault,"Moscow, Moskva, Russia",False,False,86.44,[Cybersecurity],True,https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/
181,2026-08-25,SCOTLAND,GBR,NaN,NaN,NaN,NaN,Fight,"West Yorkshire, United Kingdom (general), United Kingdom",False,False,83.80,[],False,https://www.mirror.co.uk/news/uk-news/minnie-merriman-arbroath-murdered-mourners-37589650
441,2026-08-25,KIEV,UKR,NaN,ROSTOV,RUS,NaN,Fight,"Nizhnekamsk, Tatarstan, Russia",False,True,82.83,[],False,http://www.russiaherald.com/news/279263945/ukraine-a-western-backed-zelensky-concentration-camp-moscow
325,2026-08-25,MACEDONIAN,MKD,NaN,NaN,NaN,NaN,Coerce,Macedonia,False,False,80.74,[],False,https://www.mentalfloss.com/history/medieval-court-cases-animals-trial
459,2026-08-25,VENEZUELA,VEN,NaN,GREENLAND,DNK,NaN,Fight,Venezuela,False,False,80.69,[Energy Security],True,https://www.salon.com/2026/08/25/mark-carney-tells-trump-he-will-never-own-canada-amid-tariffs/
225,2026-08-25,BRITAIN,GBR,NaN,UKRAINE,UKR,NaN,Fight,United Kingdom,False,False,80.42,[Cybersecurity],True,https://www.lbc.co.uk/article/kremlin-adviser-suggest-uk-factories-could-face-attack-from-unknown-sources-5HjdgTK_2/


In [134]:
from urllib.parse import urlparse, unquote
import re


def url_to_text(url):
    """
    Converts a news URL path into a text proxy
    that can be used for high-precision keyword matching.
    """

    if not isinstance(url, str):
        return ""

    path = unquote(urlparse(url).path)

    text = re.sub(
        r"[-_/]+",
        " ",
        path
    )

    text = re.sub(
        r"\b\d+\b",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().lower()

In [135]:
df_security["url_text"] = (
    df_security["source_url"]
    .apply(url_to_text)
)

In [136]:
df_security[
    [
        "source_url",
        "url_text",
        "attention_score"
    ]
].sort_values(
    "attention_score",
    ascending=False
).head(10)

,source_url,url_text,attention_score
96,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz,category world germany to reveal who s behind attempted airport drone attack soon says merz,93.31
180,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/,news . every hours new rape figures glasgow revealed,91.10
233,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/,national headline did the uk police do this to paint henry nowak as a racist,89.82
370,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,88.28
422,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,88.28
435,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,88.28
384,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,87.60
354,https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/,article uk faces growing threat of russian proxy attacks as defence chief warns putin is 5hjdgtd,86.44
442,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,86.41
385,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407,en russia ukraine war at least killed injured in russia ukraine overnight strikes,86.41


In [137]:
URL_SECURITY_KEYWORDS = {

    "Defence & Military": [
        "military",
        "armed forces",
        "air defence",
        "air defense",
        "missile",
        "missile strike",
        "drone strike",
        "drone strikes",
        "artillery",
        "troops",
        "nato",
        "defence ministry",
        "defense ministry"
    ],

    "Cybersecurity": [
        "cyberattack",
        "cyber attack",
        "cybersecurity",
        "ransomware",
        "malware",
        "hacking",
        "data breach"
    ],

    "Energy Security": [
        "energy security",
        "gas pipeline",
        "oil pipeline",
        "natural gas",
        "energy infrastructure",
        "power grid",
        "oil and gas"
    ],

    "Sanctions & Economic Security": [
        "sanctions",
        "economic sanctions",
        "export controls",
        "asset freeze",
        "embargo"
    ],

    "Conflict & Geopolitical Tensions": [
        "armed conflict",
        "military escalation",
        "invasion",
        "ceasefire",
        "hostilities",
        "airstrike",
        "air strike",
        "missile strike",
        "drone strike",
        "drone strikes",
        "shelling",
        "frontline",
        "front line"
    ]
}

In [138]:
def classify_url_security(text):

    if not isinstance(text, str):
        return []

    detected = []

    for domain, keywords in URL_SECURITY_KEYWORDS.items():

        for keyword in keywords:

            if keyword in text:
                detected.append(domain)
                break

    return detected

In [139]:
df_security["url_security_domains"] = (
    df_security["url_text"]
    .apply(classify_url_security)
)

In [140]:
df_security["url_countries"] = (
    df_security["url_text"]
    .apply(extract_countries)
)

In [141]:
def black_sea_security_context(row):

    text = row["url_text"]
    countries = row["url_countries"]

    if "black sea" not in text:
        return False

    strategic_black_sea_countries = {
        "Ukraine",
        "Russia",
        "Türkiye"
    }

    return bool(
        strategic_black_sea_countries
        .intersection(set(countries))
    )

In [142]:
df_security["black_sea_context"] = (
    df_security.apply(
        black_sea_security_context,
        axis=1
    )
)

In [143]:
def classify_security_domains_v3(row):

    themes = set(row["theme_list"])

    url_domains = set(
        row["url_security_domains"]
    )

    strategic_context = row["strategic_context"]

    black_sea_context = row["black_sea_context"]

    root_code = row["event_root_code"]

    domains = []


    # ----------------------------------
    # CYBERSECURITY
    # ----------------------------------

    if (
        themes.intersection(CYBER_THEMES)
        or "Cybersecurity" in url_domains
    ):
        domains.append(
            "Cybersecurity"
        )


    # ----------------------------------
    # SANCTIONS
    # ----------------------------------

    if (
        themes.intersection(SANCTIONS_THEMES)
        or "Sanctions & Economic Security"
        in url_domains
    ):
        domains.append(
            "Sanctions & Economic Security"
        )


    # ----------------------------------
    # DEFENCE & MILITARY
    # ----------------------------------

    defence_theme = bool(
        themes.intersection(
            DIRECT_DEFENCE_THEMES
            | BROAD_DEFENCE_THEMES
        )
    )

    defence_url = (
        "Defence & Military"
        in url_domains
    )

    if (
        defence_url
        or (
            defence_theme
            and (
                strategic_context
                or black_sea_context
            )
        )
    ):
        domains.append(
            "Defence & Military"
        )


    # ----------------------------------
    # CONFLICT / GEOPOLITICAL
    # ----------------------------------

    conflict_theme = bool(
        themes.intersection(
            DIRECT_CONFLICT_THEMES
            | BROAD_CONFLICT_THEMES
        )
    )

    conflict_url = (
        "Conflict & Geopolitical Tensions"
        in url_domains
    )

    conflict_event = (
        root_code
        in SECURITY_CONFLICT_ROOTS
    )

    if (
        conflict_url
        or (
            conflict_theme
            and conflict_event
            and (
                strategic_context
                or black_sea_context
            )
        )
    ):
        domains.append(
            "Conflict & Geopolitical Tensions"
        )


    # ----------------------------------
    # ENERGY SECURITY
    # ----------------------------------

    energy_theme = bool(
        themes.intersection(
            ENERGY_THEMES
        )
    )

    energy_url = (
        "Energy Security"
        in url_domains
    )

    if (
        energy_url
        or (
            energy_theme
            and (
                strategic_context
                or black_sea_context
                or bool(
                    themes.intersection(
                        SANCTIONS_THEMES
                        | BROAD_CONFLICT_THEMES
                        | DIRECT_CONFLICT_THEMES
                    )
                )
            )
        )
    ):
        domains.append(
            "Energy Security"
        )


    return list(dict.fromkeys(domains))

In [144]:
df_security["security_domains_v3"] = (
    df_security.apply(
        classify_security_domains_v3,
        axis=1
    )
)

df_security["security_relevant_v3"] = (
    df_security["security_domains_v3"]
    .apply(lambda x: len(x) > 0)
)

In [145]:
print(
    "V1 relevant:",
    df_security["security_relevant"].sum()
)

print(
    "V2 relevant:",
    df_security["security_relevant_v2"].sum()
)

print(
    "V3 relevant:",
    df_security["security_relevant_v3"].sum()
)

V1 relevant: 261
V2 relevant: 168
V3 relevant: 164


In [146]:
relevance_v3_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor2",
            "actor2_country",
            "actor2_type",
            "event_root_label",
            "location",
            "attention_score",
            "strategic_context",
            "url_text",
            "url_security_domains",
            "security_domains_v3",
            "security_relevant_v3",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_v3_check

,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_root_label,location,attention_score,strategic_context,url_text,url_security_domains,security_domains_v3,security_relevant_v3,source_url
96,2026-08-25,GERMANY,DEU,NaN,NaN,NaN,NaN,Fight,"Berlin, Berlin, Germany",93.31,False,category world germany to reveal who s behind attempted airport drone attack soon says merz,[],[],False,https://www.freemalaysiatoday.com/category/world/2026/08/25/germany-to-reveal-who-s-behind-attempted-airport-drone-attack-soon-says-merz
180,2026-08-25,GLASGOW,GBR,NaN,NaN,NaN,NaN,Assault,"Glasgow, Glasgow City, United Kingdom",91.10,False,news . every hours new rape figures glasgow revealed,[],[],False,https://www.glasgowtimes.co.uk/news/26492585.1-every-18-hours--new-rape-figures-glasgow-revealed/
233,2026-08-25,UNITED KINGDOM,GBR,COP,NaN,NaN,NaN,Fight,"Hampshire, Hampshire, United Kingdom",89.82,False,national headline did the uk police do this to paint henry nowak as a racist,[],[],False,https://wcbm.com/national-headline/did-the-uk-police-do-this-to-paint-henry-nowak-as-a-racist/
370,2026-08-25,RUSSIAN,RUS,NaN,MAYOR,NaN,GOV,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",88.28,True,en russia ukraine war at least killed injured in russia ukraine overnight strikes,[],"[Defence & Military, Conflict & Geopolitical Tensions, Energy Security]",True,https://aa.com.tr/en/russia-ukraine-war/at-least-3-killed-15-injured-in-russia-ukraine-overnight-strikes/4036407
354,2026-08-25,RUSSIAN,RUS,NaN,NaN,NaN,NaN,Assault,"Moscow, Moskva, Russia",86.44,False,article uk faces growing threat of russian proxy attacks as defence chief warns putin is 5hjdgtd,[],[Cybersecurity],True,https://www.lbc.co.uk/article/uk-faces-growing-threat-of-russian-proxy-attacks-as-defence-chief-warns-putin-is-5HjdgTD_2/
181,2026-08-25,SCOTLAND,GBR,NaN,NaN,NaN,NaN,Fight,"West Yorkshire, United Kingdom (general), United Kingdom",83.80,False,news uk news minnie merriman arbroath murdered mourners,[],[],False,https://www.mirror.co.uk/news/uk-news/minnie-merriman-arbroath-murdered-mourners-37589650
441,2026-08-25,KIEV,UKR,NaN,ROSTOV,RUS,NaN,Fight,"Nizhnekamsk, Tatarstan, Russia",82.83,True,news ukraine a western backed zelensky concentration camp moscow,[],[],False,http://www.russiaherald.com/news/279263945/ukraine-a-western-backed-zelensky-concentration-camp-moscow
325,2026-08-25,MACEDONIAN,MKD,NaN,NaN,NaN,NaN,Coerce,Macedonia,80.74,False,history medieval court cases animals trial,[],[],False,https://www.mentalfloss.com/history/medieval-court-cases-animals-trial
459,2026-08-25,VENEZUELA,VEN,NaN,GREENLAND,DNK,NaN,Fight,Venezuela,80.69,False,mark carney tells trump he will never own canada amid tariffs,[],[Energy Security],True,https://www.salon.com/2026/08/25/mark-carney-tells-trump-he-will-never-own-canada-amid-tariffs/
225,2026-08-25,BRITAIN,GBR,NaN,UKRAINE,UKR,NaN,Fight,United Kingdom,80.42,False,article kremlin adviser suggest uk factories could face attack from unknown sources 5hjdgtk,[],[Cybersecurity],True,https://www.lbc.co.uk/article/kremlin-adviser-suggest-uk-factories-could-face-attack-from-unknown-sources-5HjdgTK_2/


## 21. Final Security-Relevant Dataset

In [147]:
df_relevant = df_security[
    df_security["security_relevant_v3"]
].copy()

print("Total European / strategic events:", len(df_security))
print("Final security-relevant events:", len(df_relevant))

Total European / strategic events: 462
Final security-relevant events: 164


In [148]:
domain_distribution_v3 = (
    df_relevant["security_domains_v3"]
    .explode()
    .value_counts()
    .reset_index()
)

domain_distribution_v3.columns = [
    "security_domain",
    "events"
]

domain_distribution_v3

,security_domain,events
0,Defence & Military,93
1,Energy Security,78
2,Cybersecurity,54
3,Conflict & Geopolitical Tensions,20
4,Sanctions & Economic Security,11


## 22. SQLite Database

In [149]:
import sqlite3

In [150]:
database_path = (
    project_root
    / "data"
    / "security_monitor.db"
)

print(database_path)

c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\security_monitor.db


In [151]:
conn = sqlite3.connect(database_path)

print("SQLite connection created")

SQLite connection created


In [152]:
["Defence & Military", "Conflict & Geopolitical Tensions"]

['Defence & Military', 'Conflict & Geopolitical Tensions']

In [153]:
df_sql = df_relevant.copy()

In [154]:
df_sql["security_domains"] = (
    df_sql["security_domains_v3"]
    .apply(lambda x: " | ".join(x))
)

In [155]:
df_sql[
    [
        "security_domains_v3",
        "security_domains"
    ]
].head()

,security_domains_v3,security_domains
0,[Energy Security],Energy Security
5,[Defence & Military],Defence & Military
6,[Defence & Military],Defence & Military
13,[Defence & Military],Defence & Military
15,[Defence & Military],Defence & Military


In [156]:
security_event_columns = [
    "event_id",
    "event_date",

    "actor1",
    "actor1_country",
    "actor1_type",

    "actor2",
    "actor2_country",
    "actor2_type",

    "event_code",
    "event_root_code",
    "event_root_label",
    "quad_class",
    "quad_class_label",

    "goldstein_scale",
    "avg_tone",

    "num_mentions",
    "num_articles",

    "location",
    "latitude",
    "longitude",

    "security_domains",

    "attention_score",
    "attention_band",

    "source_name",
    "source_url"
]

In [157]:
df_sql_events = df_sql[
    security_event_columns
].copy()

In [158]:
print(df_sql_events.shape)

df_sql_events.head()

(164, 25)


,event_id,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_code,event_root_code,...,num_mentions,num_articles,location,latitude,longitude,security_domains,attention_score,attention_band,source_name,source_url
0,1319838769,2016-08-27,SRI LANKA,LKA,NaN,INVESTOR,NaN,BUS,13,1,...,4,4,Russia,60.0000,100.0000,Energy Security,26.43,Low,freemalaysiatoday.com,https://www.freemalaysiatoday.com/category/business/2026/08/25/sri-lanka-opens-offshore-oil-and-gas-blocks-to-global-bidders
5,1319838776,2025-08-25,NAVY,NaN,MIL,RUSSIAN,RUS,NaN,90,9,...,7,7,"Kremlin, Moskva, Russia",55.7522,37.6156,Defence & Military,37.91,Medium,express.co.uk,https://www.express.co.uk/news/world/2242228/zelensky-gives-britain-intelligence-goldmine-putin-ukraine
6,1319838777,2025-08-25,NAVY,NaN,MIL,RUSSIAN,RUS,NaN,90,9,...,1,1,United Kingdom,54.0000,-4.0000,Defence & Military,26.35,Low,express.co.uk,https://www.express.co.uk/news/world/2242228/zelensky-gives-britain-intelligence-goldmine-putin-ukraine
13,1319838795,2026-08-18,IRELAND,IRL,NaN,UNITED KINGDOM,GBR,NaN,40,4,...,1,1,"Portlaoise, Laois, Ireland",53.0322,-7.3000,Defence & Military,30.64,Low,irishnews.com,https://www.irishnews.com/news/northern-ireland/sinn-feins-john-finucane-wont-confirm-if-he-met-with-daniel-kinahan-and-compares-it-to-asking-a-surgeon-which-patients-theyd-spoken-to-GDLQQ7CUDZGJ7FY6LIW76BXEUQ/
15,1319838797,2026-08-18,IRELAND,IRL,COP,UNITED KINGDOM,GBR,NaN,40,4,...,4,4,"Portlaoise, Laois, Ireland",53.0322,-7.3000,Defence & Military,38.29,Medium,irishnews.com,https://www.irishnews.com/news/northern-ireland/sinn-feins-john-finucane-wont-confirm-if-he-met-with-daniel-kinahan-and-compares-it-to-asking-a-surgeon-which-patients-theyd-spoken-to-GDLQQ7CUDZGJ7FY6LIW76BXEUQ/


In [159]:
print("Rows:", len(df_sql_events))
print("Columns:", len(df_sql_events.columns))
print("Database:", database_path)

Rows: 164
Columns: 25
Database: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\security_monitor.db


## 23. Store Security Events in SQLite

In [160]:
if_exists="replace"

In [161]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

tables

,name
0,update_history
1,security_events


In [162]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("Table 'security_events' created successfully")

Table 'security_events' created successfully


In [163]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

tables

,name
0,update_history
1,security_events


In [164]:
row_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_events
    FROM security_events;
    """,
    conn
)

row_count

,total_events
0,164


## 24. SQL Analysis

In [165]:
query = """
SELECT
    attention_band,
    COUNT(*) AS events
FROM security_events
GROUP BY attention_band
ORDER BY events DESC;
"""

attention_sql = pd.read_sql_query(
    query,
    conn
)

attention_sql

,attention_band,events
0,Low,71
1,Medium,59
2,Critical,19
3,High,15


In [166]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    security_domains,
    attention_score,
    attention_band
FROM security_events
ORDER BY attention_score DESC
LIMIT 10;
"""

top_events_sql = pd.read_sql_query(
    query,
    conn
)

top_events_sql

,event_date,actor1,actor2,event_root_label,location,security_domains,attention_score,attention_band
0,2026-08-25 00:00:00,RUSSIAN,MAYOR,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,88.28,Critical
1,2026-08-25 00:00:00,UKRAINE,NATIONAL POLICE,Fight,"Rostov, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,88.28,Critical
2,2026-08-25 00:00:00,UKRAINE,MAYOR,Fight,"Rostov, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,88.28,Critical
3,2026-08-25 00:00:00,RUSSIAN,UKRAINE,Assault,"Rostov, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,87.60,Critical
4,2026-08-25 00:00:00,RUSSIAN,NaN,Assault,"Moscow, Moskva, Russia",Cybersecurity,86.44,Critical
5,2026-08-25 00:00:00,RUSSIAN,MOLDOVA,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,86.41,Critical
6,2026-08-25 00:00:00,RUSSIAN,KHARKIV,Fight,"Rostov, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,86.41,Critical
7,2026-08-25 00:00:00,UKRAINE,RUSSIAN,Fight,"Rostov, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,86.41,Critical
8,2026-08-25 00:00:00,RUSSIA,NaN,Coerce,"Rostov, Rostovskaya Oblast', Russia",Energy Security,84.72,Critical
9,2026-08-25 00:00:00,UKRAINE,RUSSIA,Fight,"Novoshakhtinsk, Rostovskaya Oblast', Russia",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,84.01,Critical


In [167]:
query = """
SELECT
    event_root_label,
    COUNT(*) AS events,
    ROUND(
        AVG(attention_score),
        2
    ) AS avg_attention_score
FROM security_events
GROUP BY event_root_label
ORDER BY events DESC;
"""

event_types_sql = pd.read_sql_query(
    query,
    conn
)

event_types_sql.head(15)

,event_root_label,events,avg_attention_score
0,Consult,37,27.05
1,Make Public Statement,26,38.82
2,Engage in Diplomatic Cooperation,19,31.49
3,Fight,16,82.57
4,Disapprove,16,51.87
5,Appeal,11,34.20
6,Express Intent to Cooperate,8,32.99
7,Yield,6,29.84
8,Provide Aid,6,30.68
9,Threaten,5,65.36


In [168]:
import plotly.express as px

print("Plotly ready")

Plotly ready


In [169]:
fig.show(renderer="browser")

NameError: name 'fig' is not defined

In [ ]:
attention_order = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

attention_sql["attention_band"] = pd.Categorical(
    attention_sql["attention_band"],
    categories=attention_order,
    ordered=True
)

attention_sql = attention_sql.sort_values(
    "attention_band"
)

In [ ]:
fig = px.bar(
    attention_sql,
    x="attention_band",
    y="events",
    title="Security Events by Attention Level",
    labels={
        "attention_band": "Attention Level",
        "events": "Number of Events"
    }
)

fig.show(renderer="browser")

In [ ]:
print(attention_sql)

check_total = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_events
    FROM security_events;
    """,
    conn
)

print(check_total)

  attention_band  events
0            Low      50
1         Medium      30
2           High      20
3       Critical      14
   total_events
0           114


In [ ]:
event_types_plot = event_types_sql.sort_values(
    "avg_attention_score",
    ascending=True
)

fig2 = px.bar(
    event_types_plot,
    x="avg_attention_score",
    y="event_root_label",
    orientation="h",
    title="Average Attention Score by Event Type",
    labels={
        "avg_attention_score": "Average Attention Score",
        "event_root_label": "Event Type"
    }
)

fig2.show(renderer="browser")

In [ ]:
query = """
SELECT
    event_root_label,
    COUNT(*) AS events,
    ROUND(
        AVG(attention_score),
        2
    ) AS avg_attention_score
FROM security_events
GROUP BY event_root_label
ORDER BY events DESC;
"""

event_types_sql = pd.read_sql_query(
    query,
    conn
)

event_types_sql

,event_root_label,events,avg_attention_score
0,Consult,28,34.05
1,Fight,20,78.63
2,Make Public Statement,12,36.31
3,Express Intent to Cooperate,12,27.24
4,Provide Aid,8,29.62
5,Engage in Diplomatic Cooperation,8,34.26
6,Coerce,7,66.93
7,Appeal,5,38.85
8,Reject,4,53.97
9,Threaten,3,66.70


In [ ]:
fig3 = px.scatter(
    event_types_sql,
    x="events",
    y="avg_attention_score",
    size="events",
    hover_name="event_root_label",
    title="Event Frequency vs Average Attention Score",
    labels={
        "events": "Number of Events",
        "avg_attention_score": "Average Attention Score"
    }
)

fig3.show(renderer="browser")

In [ ]:
fig3 = px.scatter(
    event_types_sql,
    x="events",
    y="avg_attention_score",
    size="events",
    hover_name="event_root_label",
    title="Event Frequency vs Average Attention Score",
    labels={
        "events": "Number of Events",
        "avg_attention_score": "Average Attention Score"
    }
)

fig3.show(renderer="browser")

## 29. Interactive Security Events Map

In [ ]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    latitude,
    longitude,
    security_domains,
    attention_score,
    attention_band,
    source_url
FROM security_events
WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL;
"""

map_data = pd.read_sql_query(
    query,
    conn
)

print("Events available for map:", len(map_data))

map_data.head()

Events available for map: 114


,event_date,actor1,actor2,event_root_label,location,latitude,longitude,security_domains,attention_score,attention_band,source_url
0,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRANCE,Make Public Statement,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
1,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRENCH,Make Public Statement,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
2,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRENCH,Make Public Statement,"Maroua, Extreme-Nord, Cameroon",10.5909,14.31590,Defence & Military,28.09,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
3,2026-08-17 00:00:00,POLAND,AMBASSADOR,Consult,"Warsaw, (PL67), Poland",52.2500,21.00000,Defence & Military,30.02,Low,https://forward.com/fast-forward/846473/israel-slammed-for-closing-probe-into-killing-of-seven-world-central-kitchen-workers/
4,2026-08-24 00:00:00,NaN,LONDON,Disapprove,"Kyiv, Kyyiv, Misto, Ukraine",50.4333,30.51670,Defence & Military,49.47,Medium,https://londonlovesbusiness.com/kremlin-warns-britain-over-burnhams-secret-storm-shadow-plans/


In [ ]:
fig4 = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    size="attention_score",
    hover_name="location",
    hover_data={
        "actor1": True,
        "actor2": True,
        "event_root_label": True,
        "security_domains": True,
        "attention_score": True,
        "attention_band": True,
        "latitude": False,
        "longitude": False
    },
    zoom=3,
    center={
        "lat": 54,
        "lon": 15
    },
    title="European Security Events Monitor"
)

fig4.update_layout(
    map_style="open-street-map"
)

fig4.show(renderer="browser")

In [ ]:
print("location_countries" in df_sql.columns)

True


In [ ]:
df_sql["location_countries_text"] = (
    df_sql["location_countries"]
    .apply(
        lambda x: " | ".join(x)
        if isinstance(x, list)
        else ""
    )
)

In [ ]:
df_sql[
    [
        "location",
        "location_countries",
        "location_countries_text"
    ]
].head(10)

,location,location_countries,location_countries_text
8,"Paris, France (general), France",[France],France
9,"Paris, France (general), France",[France],France
10,"Maroua, Extreme-Nord, Cameroon",[],
11,"Warsaw, (PL67), Poland",[Poland],Poland
18,"Kyiv, Kyyiv, Misto, Ukraine",[Ukraine],Ukraine
20,Turkey,[Türkiye],Türkiye
21,Ukraine,[Ukraine],Ukraine
22,"Odesa, Odes'ka Oblast, Ukraine",[Ukraine],Ukraine
30,"White House, District of Columbia, United States",[],
31,"White House, District of Columbia, United States",[],


In [ ]:
df_sql_events = df_sql[
    security_event_columns
].copy()

print("Rows:", len(df_sql_events))
print("Columns:", len(df_sql_events.columns))

Rows: 114
Columns: 25


In [ ]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite table updated")

SQLite table updated


In [ ]:
df_sql["location_countries_text"]

8       France
9       France
10            
11      Poland
18     Ukraine
        ...   
258    Ukraine
263    Ukraine
264    Ukraine
265    Ukraine
266           
Name: location_countries_text, Length: 114, dtype: str

In [ ]:
security_event_columns = [
    "event_id",
    "event_date",

    "actor1",
    "actor1_country",
    "actor1_type",

    "actor2",
    "actor2_country",
    "actor2_type",

    "event_code",
    "event_root_code",
    "event_root_label",
    "quad_class",
    "quad_class_label",

    "goldstein_scale",
    "avg_tone",

    "num_mentions",
    "num_articles",

    "location",
    "location_countries_text",
    "latitude",
    "longitude",

    "security_domains",

    "attention_score",
    "attention_band",

    "source_name",
    "source_url"
]

In [ ]:
df_sql_events = df_sql[
    security_event_columns
].copy()

In [ ]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite table updated")
print("Rows:", len(df_sql_events))

SQLite table updated
Rows: 114


In [ ]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    location_countries_text,
    latitude,
    longitude,
    security_domains,
    attention_score,
    attention_band,
    source_url
FROM security_events
WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND location_countries_text IS NOT NULL
  AND location_countries_text <> '';
"""

map_data = pd.read_sql_query(
    query,
    conn
)

print("Events available for European map:", len(map_data))

Events available for European map: 86


In [ ]:
fig4 = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    size="attention_score",
    hover_name="location",
    hover_data={
        "actor1": True,
        "actor2": True,
        "event_root_label": True,
        "security_domains": True,
        "attention_score": True,
        "attention_band": True,
        "latitude": False,
        "longitude": False
    },
    zoom=3,
    center={
        "lat": 54,
        "lon": 15
    },
    title="European Security Events Monitor"
)

fig4.update_layout(
    map_style="open-street-map"
)

fig4.show(renderer="browser")

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    "../data/security_monitor.db"
)

In [ ]:
update_history = pd.read_sql_query(
    """
    SELECT *
    FROM update_history
    ORDER BY update_time DESC;
    """,
    conn
)

update_history.head(10)

,update_time,gdelt_batch,events_processed,geographic_events,security_relevant_events,new_events_added,events_refreshed,database_total_events
0,2026-08-25 09:38:15 UTC,20260825094500,893,161,64,64,0,170
